# Proyecto Final: Clasificación de demencia con ADNI

**Nombre:** Victor Francisco Duarte Villalobos

**Matrícula:** 403230

**Programa académico:** Maestría en Ingeniería en Computación

**Materia:** Machine Learning

**Actividad:** Proyecto Final — Comparación de modelos de clasificación y métodos de ensamble

**Docente:** Dra. Olanda Prieto Ordaz

**Fecha:** 19 de noviembre de 2026

# II. Introducción

En este proyecto se reúnen todos los modelos de clasificación vistos en la materia para resolver un mismo problema: distinguir entre personas cognitivamente normales (CN) y personas con demencia (Dementia) a partir de datos demográficos, genéticos (APOE4) y volúmenes cerebrales obtenidos por resonancia magnética, usando la base de datos ADNI.

El notebook es acumulativo: conserva los modelos de las prácticas anteriores (Dummy, Regresión Logística, SVM y Random Forest) y agrega los que faltaban: `SGDClassifier` y los métodos de ensamble por *boosting* (AdaBoost, Gradient Boosting y XGBoost), por votación (Voting duro y suave) y por apilamiento (Stacking). Todos se evalúan con los mismos datos, el mismo preprocesamiento y las mismas particiones, para identificar el modelo con mejor desempeño.

- **Proyecto:** clasificación de demencia a partir de biomarcadores de resonancia magnética.
- **Dataset:** ADNI (Alzheimer's Disease Neuroimaging Initiative), una fila por paciente en su primera visita.
- **Variable objetivo:** diagnóstico `DX` (1 = Dementia, 0 = CN).
- **Tipo de tarea:** clasificación binaria supervisada.
- **Reproducibilidad:** semilla `random_state = 42` en la partición, la validación cruzada y los modelos (`random_state = 0` en la validación cruzada repetida). Se ejecuta en Google Colab con los archivos `ADNI_practica.csv` y `APOE4_C2N.csv` en Google Drive; las versiones de las librerías se imprimen al inicio.
- **Objetivo:** comparar todos los modelos de la materia, optimizar sus hiperparámetros, seleccionar el mejor con un criterio definido antes de ver Test, explicar sus predicciones con SHAP y exportar los modelos entrenados para la interfaz en Streamlit.

| Parte | Contenido |
|---|---|
| III. Preparación de datos (Fases 1 a 3) | Carga, APOE4 (C2N + MICE), partición, EDA, ingeniería de características y Pipeline |
| IV. Experimentación (Fases 4 a 7) | Modelos con valores por defecto, optimización, ensambles, selección del modelo final y ajuste del umbral |
| V. Evaluación final (Fases 8 a 10) | Evaluación en Test, comparación de todos los modelos e interpretabilidad con SHAP |
| VI a VIII | Comparación con la literatura, conclusiones y referencias |
| IX. Anexo | Exportación de los modelos entrenados para la interfaz |

> **El conjunto Test se separa antes del EDA y nunca se usa para elegir modelo, hiperparámetros, umbral ni decisiones de preprocesamiento.**

# III. Preparación de datos

# Fase 1. Preparación de los datos

## 1. Descripción del problema

- **Nombre del dataset:** ADNI (Alzheimer's Disease Neuroimaging Initiative), fases ADNI1, GO, 2, 3 y 4.
- **Fuente:** LONI Image & Data Archive (IDA), paquete ADNIMERGE2. Tablas utilizadas: `ADSP_PHC_DEMODX` (diagnóstico y demografía), `ADSP_PHC_T1_FS` (volúmenes cerebrales de FreeSurfer armonizados con ComBat), `APOERES` (genotipo APOE) y `C2N_PRECIVITYAD2_PLASMA` (APOE determinado en plasma).
- **Número de instancias:** 1,624 pacientes (1,224 CN y 400 Dementia).
- **Número de características:** 18 predictoras originales: edad, sexo, escolaridad, número de copias de APOE4, volumen intracraneal (ICV) y 13 medidas cerebrales (hipocampo, corteza entorrinal, amígdala, ventrículos laterales y ventrículos laterales inferiores de cada hemisferio, grosor cortical medio de cada hemisferio y volumen cortical).
- **Variable objetivo:** `DX` (1 = Dementia, 0 = CN).
- **Tipo de problema:** clasificación binaria.
- **Objetivo predictivo:** identificar a los pacientes con demencia a partir de su resonancia magnética, su genotipo APOE4 y sus datos demográficos, sin usar pruebas cognitivas (MMSE, CDR o ADAS), ya que el diagnóstico se basa en ellas.

In [ ]:
# XGBoost y SHAP vienen instalados en Colab; si faltan, se instalan
import importlib.util
import subprocess
import sys

for paquete in ['xgboost', 'shap']:
    if importlib.util.find_spec(paquete) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', paquete], check=True)

In [ ]:
import os
import sys
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_validate,
                                     cross_val_predict, GridSearchCV, RepeatedStratifiedKFold)
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier,
                              VotingClassifier, StackingClassifier)
from xgboost import XGBClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (make_scorer, accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve,
                             precision_recall_curve, average_precision_score)
from sklearn.exceptions import ConvergenceWarning

import joblib
import shap
import sklearn
import xgboost

warnings.filterwarnings('ignore', category=ConvergenceWarning)
# Avisos de cambios en versiones futuras de las librerías (también en los procesos paralelos)
warnings.filterwarnings('ignore', category=FutureWarning)
os.environ['PYTHONWARNINGS'] = 'ignore::FutureWarning'
pd.set_option('display.max_colwidth', None)

RANDOM_STATE = 42

# Estilo de las gráficas
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': '#e6e5e0',
    'grid.linewidth': 0.8,
    'axes.axisbelow': True,
    'axes.edgecolor': '#8a8984',
    'axes.labelcolor': '#52514e',
    'xtick.color': '#52514e',
    'ytick.color': '#52514e',
    'axes.titleweight': 'bold',
    'axes.titlesize': 12,
    'font.size': 10,
})
COLOR_CN = '#2a78d6'
COLOR_DEM = '#eb6834'
COLORES = ['#2a78d6', '#eb6834', '#1baf7a']
COLOR_TEXTO = '#52514e'

print('Python', sys.version.split()[0], '| scikit-learn', sklearn.__version__, '| xgboost', xgboost.__version__,
      '| shap', shap.__version__, '| numpy', np.__version__, '| pandas', pd.__version__)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUTA = '/content/drive/MyDrive/'

# Las figuras principales se guardan en alta resolución para el reporte técnico
CARPETA_FIGURAS = RUTA + 'figuras_proyecto/'
os.makedirs(CARPETA_FIGURAS, exist_ok=True)


def guardar_figura(fig, nombre):
    fig.savefig(CARPETA_FIGURAS + nombre + '.png', dpi=300, bbox_inches='tight')

In [ ]:
df_original = pd.read_csv(RUTA + 'ADNI_practica.csv')

print('Dataset completo:', df_original.shape)
print(df_original['DX'].value_counts())

# Problema de clasificación: CN vs Dementia
df = df_original[df_original['DX'].isin(['CN', 'Dementia'])].copy()
df = df.reset_index(drop=True)

print('\nDataset del proyecto:', df.shape)
print(df['DX'].value_counts())

### Cambio respecto a la Práctica 1: APOE4

En la Práctica 1, los pacientes sin genotipo APOE4 se imputaban con la moda, es decir, se les asignaban 0 copias aunque tuvieran demencia. Para este proyecto se buscó el dato real en otras tablas de ADNI:

- `C2N_PRECIVITYAD2_PLASMA` determina el genotipo APOE a partir de una muestra de sangre. En los 860 pacientes que tienen tanto genotipo (`APOERES`) como prueba en plasma, ambos coinciden en el número de copias de APOE4 en **855 casos (99.4%)**, y nunca se confunden las 2 copias.
- Con esta tabla se recuperan **44 pacientes**. APOE es genético y no cambia con el tiempo, y no se deriva del diagnóstico, por lo que no genera fuga de información.
- Los pacientes que siguen sin dato se imputan con **MICE** dentro del Pipeline (sección 6).

In [ ]:
apoe_c2n = pd.read_csv(RUTA + 'APOE4_C2N.csv')

df = df.merge(apoe_c2n, on='RID', how='left')
df['Fuente_APOE4'] = np.where(df['APOE4'].notnull(), 'Genotipo (APOERES)',
                     np.where(df['APOE4_C2N'].notnull(), 'Plasma (C2N)', 'Sin dato (MICE)'))
df['APOE4'] = df['APOE4'].fillna(df['APOE4_C2N'])
df = df.drop(columns='APOE4_C2N')

print('Origen del dato de APOE4:')
print(df['Fuente_APOE4'].value_counts())

## 2. Inspección inicial

Se revisan dimensiones, nombres de columnas, tipos de datos, primeras observaciones, valores faltantes, registros duplicados, identificadores y la distribución general de la variable objetivo. En esta etapa no se aplican transformaciones dependientes de los datos.

In [ ]:
print('Dimensiones (instancias, columnas):', df.shape)
print('\nNombres de las columnas:')
print(df.columns.tolist(), '\n')
df.info()

In [ ]:
df.head()

In [ ]:
faltantes = df.isnull().sum()
print('Valores faltantes por columna:')
print(faltantes[faltantes > 0])
print('\nRegistros duplicados:', df.duplicated().sum())
print('Pacientes únicos (RID):', df['RID'].nunique(), 'de', len(df), 'filas')

In [ ]:
# Corrección de tipo: las fechas vienen como texto
df['EXAMDATE'] = pd.to_datetime(df['EXAMDATE'])
df['PHC_SCANDATE'] = pd.to_datetime(df['PHC_SCANDATE'])

identificadores = ['RID', 'VISCODE2', 'EXAMDATE', 'PHC_SCANDATE', 'Fuente_APOE4']
print('Columnas que no deben usarse como predictores:', identificadores)

print('\nDistribución general de la variable objetivo:')
print(pd.DataFrame({'Conteo': df['DX'].value_counts(),
                    'Proporción': df['DX'].value_counts(normalize=True).round(3)}))

In [ ]:
origen = df['Fuente_APOE4'].value_counts()[['Genotipo (APOERES)', 'Plasma (C2N)', 'Sin dato (MICE)']]

fig, ax = plt.subplots(figsize=(8, 3.2))
barras = ax.barh(origen.index[::-1], origen.values[::-1], color=COLORES[::-1], height=0.6)
for barra, valor in zip(barras, origen.values[::-1]):
    ax.text(barra.get_width() + 15, barra.get_y() + barra.get_height() / 2,
            f'{valor:,}  ({valor / len(df):.1%})', va='center', color=COLOR_TEXTO)
ax.set_xlim(0, origen.max() * 1.25)
ax.set_xlabel('Número de pacientes')
ax.grid(axis='y', visible=False)
ax.set_title('Origen del dato de APOE4')
plt.tight_layout()
plt.show()

# Fase 2. Partición y EDA

## 3. Definición de X e y

Variables eliminadas de X:

- `RID`: identificador del paciente; no describe al paciente y el modelo podría memorizarlo.
- `VISCODE2`: código de la visita; es un dato administrativo, no clínico.
- `EXAMDATE` y `PHC_SCANDATE`: fechas de la visita y de la resonancia; reflejan cuándo ingresó el paciente al estudio, no su estado de salud.
- `Fuente_APOE4`: solo documenta de dónde se obtuvo APOE4.
- `DX`: es la variable objetivo.

El dataset no incluye puntajes cognitivos (MMSE, CDR, ADAS) porque el diagnóstico se basa en ellos y su uso sería fuga de información.

In [ ]:
# Variable objetivo: 1 = Dementia, 0 = CN
y = (df['DX'] == 'Dementia').astype(int)

columnas_fuera = identificadores + ['DX']
X = df.drop(columns=columnas_fuera)

print('Dimensión de X:', X.shape)
print('Dimensión de y:', y.shape)
print('Características:', X.columns.tolist())
print('¿DX está en X?', 'DX' in X.columns)

## 4. Partición Train/Test

80% Train y 20% Test, estratificada por la variable objetivo para que ambos conjuntos conserven la proporción de pacientes con demencia.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('Train:', X_train.shape, '| Test:', X_test.shape)
print('\nProporción de Dementia')
print('  Total:', round(y.mean(), 3))
print('  Train:', round(y_train.mean(), 3))
print('  Test: ', round(y_test.mean(), 3))

## 5. Análisis Exploratorio de Datos (solo Train)

Síntesis del EDA de la Práctica 1

In [ ]:
eda = X_train.copy()
eda['DX'] = y_train.map({0: 'CN', 1: 'Dementia'})

### 5.1 Distribución de la variable objetivo y balance de clases

In [ ]:
conteo = eda['DX'].value_counts()[['CN', 'Dementia']]

fig, ax = plt.subplots(figsize=(6, 4))
barras = ax.bar(conteo.index, conteo.values, color=[COLOR_CN, COLOR_DEM], width=0.55)
for barra, valor in zip(barras, conteo.values):
    ax.text(barra.get_x() + barra.get_width() / 2, barra.get_height() + 15,
            f'{valor}  ({valor / conteo.sum():.1%})', ha='center', color=COLOR_TEXTO)
ax.set_ylim(0, conteo.max() * 1.15)
ax.set_ylabel('Número de pacientes')
ax.grid(axis='x', visible=False)
ax.set_title('Distribución de diagnósticos (Train)')
plt.tight_layout()
plt.show()

print('Accuracy de un modelo que siempre predice CN:', round(1 - y_train.mean(), 3))

### 5.2 Valores faltantes

In [ ]:
faltan_train = X_train.isnull().sum()
print(pd.DataFrame({'Faltantes': faltan_train[faltan_train > 0],
                    'Porcentaje': (faltan_train[faltan_train > 0] / len(X_train) * 100).round(1)}))

### 5.3 Variables numéricas y posibles outliers

In [ ]:
num_cols = X_train.select_dtypes(include='number').columns

# Promedio de cada variable por diagnóstico
print(eda.groupby('DX')[list(num_cols)].mean().T.round(2))

# Valores fuera de 1.5 * IQR por variable
q1, q3 = X_train[num_cols].quantile(0.25), X_train[num_cols].quantile(0.75)
iqr = q3 - q1
fuera = ((X_train[num_cols] < q1 - 1.5 * iqr) | (X_train[num_cols] > q3 + 1.5 * iqr)).sum()
print('\nValores fuera de 1.5 IQR:')
print(fuera[fuera > 0].sort_values(ascending=False))

### 5.4 Variables categóricas y discretas

In [ ]:
print('Proporción de Dementia por Sexo:')
print(pd.crosstab(eda['Sexo'], eda['DX'], normalize='index').round(3), '\n')

print('Proporción de Dementia por copias de APOE4:')
print(pd.crosstab(eda['APOE4'], eda['DX'], normalize='index').round(3))

### 5.5 Correlaciones y relación con la variable objetivo

In [ ]:
corr_y = X_train[num_cols].corrwith(y_train).sort_values()
print('Correlación de cada variable con Dementia:')
print(corr_y.round(3))

corr = X_train[num_cols].corr()
mascara = np.triu(np.ones(corr.shape), k=1).astype(bool)
pares = corr.where(mascara).stack().sort_values(key=abs, ascending=False)
print('\nPares de variables más correlacionados:')
print(pares.head(6).round(3))

# Matriz de correlación (Train), incluyendo el diagnóstico como última variable
nombres_cortos = {
    'Edad': 'Edad', 'Escolaridad': 'Escolaridad', 'ICV': 'ICV', 'APOE4': 'APOE4',
    'Left_Hippocampus': 'Hipocampo izq.', 'Right_Hippocampus': 'Hipocampo der.',
    'lh_entorhinal_volume': 'Entorrinal izq.', 'rh_entorhinal_volume': 'Entorrinal der.',
    'Left_Lateral_Ventricle': 'Ventrículo lat. izq.', 'Right_Lateral_Ventricle': 'Ventrículo lat. der.',
    'Left_Inf_Lat_Vent': 'Ventrículo inf. izq.', 'Right_Inf_Lat_Vent': 'Ventrículo inf. der.',
    'Left_Amygdala': 'Amígdala izq.', 'Right_Amygdala': 'Amígdala der.',
    'lh_MeanThickness_thickness': 'Grosor cortical izq.', 'rh_MeanThickness_thickness': 'Grosor cortical der.',
    'CortexVol': 'Volumen cortical',
}
datos_corr = X_train[num_cols].rename(columns=nombres_cortos)
datos_corr['Dementia'] = y_train
matriz = datos_corr.corr()
triangulo = np.triu(np.ones(matriz.shape, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(matriz.values, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, label='Correlación de Pearson', shrink=0.8)
ax.set_xticks(range(len(matriz)))
ax.set_xticklabels(matriz.columns, rotation=60, ha='right')
ax.set_yticks(range(len(matriz)))
ax.set_yticklabels(matriz.columns)
ax.grid(False)
for i in range(len(matriz)):
    for j in range(len(matriz)):
        v = matriz.values[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7,
                color='white' if abs(v) > 0.6 else '#0b0b0b')
ax.axhline(len(matriz) - 1.5, color=COLOR_TEXTO, linewidth=1)
ax.axvline(len(matriz) - 1.5, color=COLOR_TEXTO, linewidth=1)
ax.set_title('Matriz de correlación (Train)')
plt.tight_layout()
plt.show()

### Pregunta

1. ¿Qué elementos del preprocesamiento de la primera práctica se reutilizaron y por qué siguen siendo apropiados para los modelos SVM?<br>
Se reutilizaron la ingeniería de características (suma de los volúmenes izquierdo y derecho, normalización por ICV y grosor cortical medio), el StandardScaler para las variables numéricas, el OneHotEncoder para Sexo y la estructura ColumnTransformer + Pipeline. Siguen siendo apropiados porque el SVM calcula distancias y productos punto entre pacientes: sin escalar, las variables con rangos grandes dominarían el kernel. Además, sumar izquierdo y derecho reduce la multicolinealidad que se observa en las correlaciones, y normalizar por ICV elimina el efecto del tamaño de la cabeza. El único cambio es la imputación de APOE4, que ahora usa datos reales de C2N y MICE en lugar de la moda.

### 5.6 Dispersión de los datos

Los pacientes de Train con sus variables numéricas reducidas a 2 dimensiones mediante PCA (análisis de componentes principales), para observar cómo se distribuyen y separan las clases.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

# Variables numéricas originales de Train: imputación y escalamiento solo para la visualización
X_eda = StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(X_train[num_cols]))
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_eda)
varianza = pca.explained_variance_ratio_
cn, dem = (y_train.values == 0), (y_train.values == 1)

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.scatter(X_pca[cn, 0], X_pca[cn, 1], s=14, alpha=0.45, color=COLOR_CN, label='CN')
ax.scatter(X_pca[dem, 0], X_pca[dem, 1], s=14, alpha=0.6, color=COLOR_DEM, label='Dementia')
ax.set_xlabel(f'Componente principal 1 ({varianza[0]:.0%} de la varianza)')
ax.set_ylabel(f'Componente principal 2 ({varianza[1]:.0%} de la varianza)')
ax.legend(frameon=False)
ax.set_title('Dispersión de los pacientes (PCA, Train)')
plt.tight_layout()
plt.show()

print(f'Varianza explicada por los 2 componentes: {varianza.sum():.1%}')

# Fase 3. Pipeline de preprocesamiento

## 6. Construcción del Pipeline

**Estructura:** Preprocesamiento + Modelo (el mismo Pipeline para todos los modelos)

- **Ingeniería de características** (misma función que en la Práctica 1): volúmenes totales normalizados por ICV y grosor cortical medio. Se calcula fila por fila, sin aprender nada de los datos, por lo que puede aplicarse antes del Pipeline sin fuga de información.
- **Imputación con MICE** (`IterativeImputer`): estima APOE4 (y los 2 faltantes de Escolaridad) a partir de las demás variables numéricas del paciente. En la comparación de métodos de la Práctica 1, MICE superó a la moda en todas las métricas (ROC-AUC 0.9483 vs 0.9471 y recall 0.7656 vs 0.7500) con un tiempo de ajuste similar.
- **StandardScaler** para todas las variables numéricas.
- **OneHotEncoder** para Sexo.
- **ColumnTransformer** para aplicar cada transformación a sus columnas.

El imputador, el scaler y el encoder no se ajustan previamente con todo Train: se ajustan dentro de cada fold de la validación cruzada.

In [ ]:
def crear_features(datos):
    datos = datos.copy()

    # Sumar izquierdo + derecho (están muy correlacionados)
    datos['Hipocampo_total'] = datos['Left_Hippocampus'] + datos['Right_Hippocampus']
    datos['Entorrinal_total'] = datos['lh_entorhinal_volume'] + datos['rh_entorhinal_volume']
    datos['Amigdala_total'] = datos['Left_Amygdala'] + datos['Right_Amygdala']
    datos['Ventriculos_total'] = datos['Left_Lateral_Ventricle'] + datos['Right_Lateral_Ventricle']
    datos['Vent_inferior_total'] = datos['Left_Inf_Lat_Vent'] + datos['Right_Inf_Lat_Vent']
    datos['Grosor_medio'] = (datos['lh_MeanThickness_thickness'] + datos['rh_MeanThickness_thickness']) / 2

    # Normalizar por tamaño de cabeza (ICV), en mm³ por cada litro de ICV
    for col in ['Hipocampo_total', 'Entorrinal_total', 'Amigdala_total',
                'Ventriculos_total', 'Vent_inferior_total', 'CortexVol']:
        datos[col + '_ICV'] = datos[col] / datos['ICV'] * 1000

    # Quitar columnas originales que ya quedaron representadas
    originales = ['Left_Hippocampus', 'Right_Hippocampus',
                  'lh_entorhinal_volume', 'rh_entorhinal_volume',
                  'Left_Amygdala', 'Right_Amygdala',
                  'Left_Lateral_Ventricle', 'Right_Lateral_Ventricle',
                  'Left_Inf_Lat_Vent', 'Right_Inf_Lat_Vent',
                  'lh_MeanThickness_thickness', 'rh_MeanThickness_thickness',
                  'Hipocampo_total', 'Entorrinal_total', 'Amigdala_total',
                  'Ventriculos_total', 'Vent_inferior_total', 'CortexVol', 'ICV']
    return datos.drop(columns=originales)


X_train_fe = crear_features(X_train)
X_test_fe = crear_features(X_test)

print('Train:', X_train.shape, '->', X_train_fe.shape)
print('Test: ', X_test.shape, '->', X_test_fe.shape)
print('Características finales:', X_train_fe.columns.tolist())

In [ ]:
cols_cat = ['Sexo']
cols_num = [c for c in X_train_fe.columns if c not in cols_cat]   # incluye APOE4

print('Numéricas (incluye APOE4):', cols_num)
print('Categóricas:', cols_cat)

# Numéricas + APOE4: imputación con MICE + escalamiento
pipe_num = Pipeline([
    ('imputar', IterativeImputer(max_iter=10, skip_complete=True, random_state=RANDOM_STATE)),
    ('escalar', StandardScaler())
])

# Sexo: OneHot (una sola columna porque es binaria)
pipe_cat = Pipeline([
    ('onehot', OneHotEncoder(drop='if_binary', handle_unknown='ignore'))
])

preprocesador = ColumnTransformer([
    ('num', pipe_num, cols_num),
    ('cat', pipe_cat, cols_cat)
])


def crear_pipeline(modelo):
    return Pipeline([
        ('prep', clone(preprocesador)),
        ('modelo', modelo)
    ])

In [ ]:
X_train_prep = clone(preprocesador).fit_transform(X_train_fe)
nombres = clone(preprocesador).fit(X_train_fe).get_feature_names_out()

print('Dimensión después del preprocesamiento:', X_train_prep.shape)
print('Faltantes después de imputar:', np.isnan(X_train_prep).sum())
pd.DataFrame(X_train_prep, columns=nombres).describe().T[['mean', 'std']].round(2)

# IV. Experimentación

# Fase 4. Comparación de modelos con valores por defecto

Todos los modelos se evalúan con validación cruzada estratificada de 5 folds sobre Train (`StratifiedKFold`, `shuffle=True`, `random_state=42`), los mismos folds de las prácticas anteriores. Se reportan Accuracy, Balanced Accuracy, Precision, Recall, F1 y ROC-AUC con su desviación estándar entre folds, y la brecha de Balanced Accuracy entre Train y validación para detectar overfitting.

El ROC-AUC solo necesita un puntaje que ordene a los pacientes: los modelos que tienen `decision_function` (Regresión Logística, SVM, SGD, AdaBoost y Gradient Boosting) usan la distancia a la frontera de decisión, y los que no la tienen (Random Forest y XGBoost) usan la probabilidad de Dementia (`predict_proba`).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

metricas = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': 'recall',
    'f1': make_scorer(f1_score, zero_division=0),
    'roc_auc': 'roc_auc'
}
# El voto duro solo entrega la clase, no un puntaje, por lo que no se le puede calcular el ROC-AUC en la CV
metricas_sin_auc = {k: v for k, v in metricas.items() if k != 'roc_auc'}

nombres_metricas = {'accuracy': 'Accuracy', 'balanced_accuracy': 'Balanced Accuracy',
                    'precision': 'Precision', 'recall': 'Recall', 'f1': 'F1', 'roc_auc': 'ROC-AUC'}

resultados_cv = {}


def evaluar_cv(modelo, nombre, envolver=True, scoring=metricas):
    # envolver=True coloca el modelo dentro del Pipeline de preprocesamiento.
    # Voting y Stacking (sección 12.9 y 12.10) ya contienen modelos con su propio Pipeline: usan envolver=False.
    estimador = crear_pipeline(modelo) if envolver else modelo
    inicio = time.perf_counter()
    s = cross_validate(estimador, X_train_fe, y_train, cv=cv, scoring=scoring, n_jobs=-1,
                       return_train_score=True)
    tiempo = time.perf_counter() - inicio
    resultados_cv[nombre] = s
    print(nombre)
    for m in scoring:
        print(f"  {nombres_metricas[m]:<18}: {s['test_' + m].mean():.4f}  (desv. {s['test_' + m].std():.4f})")
    brecha = s['train_balanced_accuracy'].mean() - s['test_balanced_accuracy'].mean()
    print(f'  Brecha Bal. Acc. (Train − validación): {brecha:.4f}')
    print(f'  Tiempo: {tiempo:.1f} s\n')

## Modelos de referencia (prácticas anteriores)

Se conservan el Dummy y las dos regresiones logísticas de la Práctica 1, ahora con el nuevo tratamiento de APOE4.

In [ ]:
evaluar_cv(DummyClassifier(strategy='most_frequent'), 'Dummy')
evaluar_cv(LogisticRegression(C=1e6, max_iter=5000), 'Logística sin reg.')
evaluar_cv(LogisticRegression(C=1.0, max_iter=5000), 'Logística L2')

## SGDClassifier

`SGDClassifier` con los parámetros por defecto (`loss="hinge"`, `penalty="l2"`, `alpha=0.0001`), excepto `random_state=42`.

In [ ]:
evaluar_cv(SGDClassifier(random_state=RANDOM_STATE), 'SGD')

## 7. Modelo SVM lineal — Baseline

`LinearSVC` con los parámetros por defecto, excepto `max_iter=10000` para asegurar la convergencia.

In [ ]:
evaluar_cv(LinearSVC(max_iter=10000, random_state=RANDOM_STATE), 'SVM lineal')

## 8. SVM con kernel polinomial

`SVC(kernel="poly")` con los parámetros por defecto (degree = 3, coef0 = 0, C = 1, gamma = "scale").

In [ ]:
evaluar_cv(SVC(kernel='poly'), 'SVM polinomial')

### Preguntas

1. ¿Mejora respecto al modelo lineal?<br>
No. Con los parámetros por defecto el kernel polinomial fue peor que el SVM lineal en casi todas las métricas: ROC-AUC 0.934 contra 0.946, recall 0.700 contra 0.766 y Balanced Accuracy 0.841 contra 0.868. Además fue el modelo menos estable, con una desviación del recall entre folds de 0.062.

2. ¿Qué podría indicar esta diferencia respecto a la relación entre las variables y la variable objetivo?<br>
Indica que la relación entre los biomarcadores y el diagnóstico no necesita una frontera cúbica. Con degree = 3 y coef0 = 0, el kernel solo usa términos de grado 3 y pierde los términos lineales, que son los que mejor explican el problema. Agregar complejidad sin necesidad hizo que el modelo se ajustara a ruido y detectara menos pacientes con demencia.

## 9. SVM con kernel RBF

`SVC(kernel="rbf")` con los parámetros por defecto (C = 1, gamma = "scale").

In [ ]:
evaluar_cv(SVC(kernel='rbf'), 'SVM RBF')

### Comparación con el SVM lineal y el kernel polinomial

El SVM RBF obtuvo prácticamente el mismo resultado que el SVM lineal (ROC-AUC 0.945 contra 0.946 y Balanced Accuracy 0.873 contra 0.868), con un recall y un F1 ligeramente mayores (0.775 y 0.832 contra 0.766 y 0.825). Las diferencias son menores que la desviación entre folds. Frente al kernel polinomial, el RBF fue claramente mejor en todas las métricas y más estable, lo que indica que una frontera suave funciona mejor que forzar interacciones de grado 3.

## Random Forest — Baseline

`RandomForestClassifier` con los parámetros por defecto (n_estimators = 100, max_depth = None, min_samples_leaf = 1, max_features = "sqrt", class_weight = None), excepto `random_state=42` para que los resultados sean reproducibles.

Se usa el mismo Pipeline que en los demás modelos. El escalamiento no cambia los resultados de un árbol, porque cada división solo compara un valor contra un umbral y escalar no cambia el orden de los pacientes; se conserva para que todos los modelos reciban exactamente los mismos datos. La imputación con MICE sí es necesaria.

In [ ]:
evaluar_cv(RandomForestClassifier(random_state=RANDOM_STATE), 'Random Forest')

## AdaBoost — Baseline

`AdaBoostClassifier` con los parámetros por defecto (n_estimators = 50, learning_rate = 1.0 y, como estimador base, un árbol de profundidad 1 o *stump*), excepto `random_state=42`.

In [ ]:
evaluar_cv(AdaBoostClassifier(random_state=RANDOM_STATE), 'AdaBoost')

## Gradient Boosting — Baseline

`GradientBoostingClassifier` con los parámetros por defecto (n_estimators = 100, learning_rate = 0.1, max_depth = 3, subsample = 1.0), excepto `random_state=42`.

In [ ]:
evaluar_cv(GradientBoostingClassifier(random_state=RANDOM_STATE), 'Gradient Boosting')

## XGBoost — Baseline

`XGBClassifier` (biblioteca `xgboost`) con los parámetros por defecto (n_estimators = 100, learning_rate = 0.3, max_depth = 6), excepto `random_state=42` y `n_jobs=1` (el paralelismo lo maneja la validación cruzada).

In [ ]:
evaluar_cv(XGBClassifier(random_state=RANDOM_STATE, n_jobs=1, eval_metric='logloss'), 'XGBoost')

### Brecha entre Train y validación

Se compara la Balanced Accuracy en Train y en validación para medir cuánto memoriza cada modelo. Random Forest se compara contra la Regresión Logística y el SVM RBF, y los tres modelos de boosting contra él.

In [ ]:
brecha_base = []
for nombre in ['Logística L2', 'SVM RBF', 'Random Forest', 'AdaBoost', 'Gradient Boosting', 'XGBoost']:
    s = resultados_cv[nombre]
    brecha_base.append({'Modelo': nombre,
                        'Bal. Acc. Train': s['train_balanced_accuracy'].mean(),
                        'Bal. Acc. validación': s['test_balanced_accuracy'].mean(),
                        'Brecha': s['train_balanced_accuracy'].mean() - s['test_balanced_accuracy'].mean()})
pd.DataFrame(brecha_base).set_index('Modelo').round(4)

### Análisis

Con los parámetros por defecto, Random Forest no mejoró respecto a la Regresión Logística ni a los SVM: obtuvo una Balanced Accuracy de 0.858, por debajo de la Regresión Logística L2 (0.868) y del SVM RBF (0.873). Sin embargo, su ROC-AUC fue prácticamente igual al de la logística (0.946 contra 0.947), es decir, ordena a los pacientes igual de bien, pero con el punto de corte por defecto detecta menos pacientes con demencia (recall de 0.747). Además, tiene una brecha de 0.142 entre Train (0.9996) y validación, debido a que con max_depth = None y min_samples_leaf = 1 cada árbol crece hasta memorizar a los pacientes de Train, por lo que se regulariza en la sección 12.4.

Los modelos de boosting tampoco superaron a los modelos lineales ni al SVM RBF: Gradient Boosting (0.864) y XGBoost (0.863) quedaron muy cerca de Random Forest, y AdaBoost fue el más bajo (0.841, con un ROC-AUC de 0.936). XGBoost es el que tiene mayor riesgo de overfitting, ya que llega a una Balanced Accuracy de 1.000 en Train (brecha de 0.137) por sus árboles de profundidad 6 y su learning_rate de 0.3, seguido de Gradient Boosting (0.100). AdaBoost, al usar *stumps*, tiene una brecha de solo 0.037.

Por su parte, SGD obtuvo prácticamente la misma Balanced Accuracy que la Regresión Logística L2 y el SVM lineal (0.867 contra 0.868), lo cual es esperable ya que también es un clasificador lineal. Sin embargo, tuvo un recall mayor (0.806) a cambio de una precisión menor y más variable entre folds (0.798 ± 0.079 contra ≈ 0.89 ± 0.04).

# Fase 5. Comparación inicial de modelos

## 10. Tabla comparativa (validación cruzada)

In [ ]:
def tabla_cv(nombres_modelos):
    filas = []
    for n in nombres_modelos:
        s = resultados_cv[n]
        fila = {'Modelo': n}
        for m in metricas:
            fila[nombres_metricas[m]] = f"{s['test_' + m].mean():.3f} ± {s['test_' + m].std():.3f}"
        filas.append(fila)
    return pd.DataFrame(filas).set_index('Modelo')


modelos_iniciales = list(resultados_cv)
tabla_cv(modelos_iniciales)

In [ ]:
graf_metricas = ['roc_auc', 'recall', 'f1']
x = np.arange(len(modelos_iniciales))
ancho = 0.26

fig, ax = plt.subplots(figsize=(14, 5))
for i, m in enumerate(graf_metricas):
    medias = [resultados_cv[n]['test_' + m].mean() for n in modelos_iniciales]
    desv = [resultados_cv[n]['test_' + m].std() for n in modelos_iniciales]
    ax.bar(x + (i - 1) * ancho, medias, ancho * 0.9, yerr=desv, capsize=3,
           color=COLORES[i], label=nombres_metricas[m],
           error_kw={'elinewidth': 1, 'ecolor': COLOR_TEXTO})
ax.set_xticks(x)
ax.set_xticklabels(modelos_iniciales, rotation=25, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Promedio en CV (5 folds)')
ax.grid(axis='x', visible=False)
ax.legend(ncol=3, loc='upper left', frameon=False)
ax.set_title('Comparación de modelos en validación cruzada')
plt.tight_layout()
guardar_figura(fig, 'comparacion_modelos_cv')
plt.show()

In [ ]:
modelos_box = [n for n in modelos_iniciales if n != 'Dummy']
datos_box = [resultados_cv[n]['test_roc_auc'] for n in modelos_box]

fig, ax = plt.subplots(figsize=(13, 4.2))
caja = ax.boxplot(datos_box, patch_artist=True, widths=0.5,
                  medianprops={'color': COLOR_TEXTO, 'linewidth': 1.5})
for parche in caja['boxes']:
    parche.set_facecolor('#cfe0f5')
    parche.set_edgecolor(COLOR_CN)
ax.set_xticks(range(1, len(modelos_box) + 1))
ax.set_xticklabels(modelos_box, rotation=25, ha='right')
ax.set_ylabel('ROC-AUC por fold')
ax.grid(axis='x', visible=False)
ax.set_title('Estabilidad del ROC-AUC entre folds')
plt.tight_layout()
plt.show()

### Forma de la frontera de decisión

Frontera de un SVM lineal, de un SVM RBF y de un Random Forest entrenados solo con las dos variables más correlacionadas con el diagnóstico (solo Train). Esta figura es ilustrativa: los modelos de la práctica usan las 11 características.

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap

# Las dos características más correlacionadas con el diagnóstico
f1, f2 = 'Hipocampo_total_ICV', 'Vent_inferior_total_ICV'
X2 = X_train_fe[[f1, f2]]
escalador_2 = StandardScaler().fit(X2)
X2_esc = escalador_2.transform(X2)
svm_lineal_2 = LinearSVC(class_weight='balanced', max_iter=10000, random_state=RANDOM_STATE).fit(X2_esc, y_train)
svm_rbf_2 = SVC(kernel='rbf', class_weight='balanced').fit(X2_esc, y_train)
rf_2 = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight='balanced',
                              random_state=RANDOM_STATE).fit(X2_esc, y_train)

x_min, x_max = X2[f1].quantile([0.005, 0.995])
y_min, y_max = X2[f2].quantile([0.005, 0.995])
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
malla = escalador_2.transform(pd.DataFrame({f1: xx.ravel(), f2: yy.ravel()}))
z_lineal = svm_lineal_2.decision_function(malla).reshape(xx.shape)
z_rbf = svm_rbf_2.decision_function(malla).reshape(xx.shape)
z_rf = rf_2.predict_proba(malla)[:, 1].reshape(xx.shape)
cn, dem = (y_train.values == 0), (y_train.values == 1)

fig, ax = plt.subplots(1, 2, figsize=(13, 5.2), sharey=True)
for eje in ax:
    eje.scatter(X2.loc[cn, f1], X2.loc[cn, f2], s=14, alpha=0.45, color=COLOR_CN)
    eje.scatter(X2.loc[dem, f1], X2.loc[dem, f2], s=14, alpha=0.6, color=COLOR_DEM)
    eje.set_xlim(x_min, x_max)
    eje.set_ylim(y_min, y_max)
    eje.set_xlabel('Hipocampo total / ICV')
ax[0].set_ylabel('Ventrículo lateral inferior / ICV')

# Izquierda: SVM
ax[0].contour(xx, yy, z_lineal, levels=[0], colors='#0b0b0b', linewidths=2)
ax[0].contour(xx, yy, z_rbf, levels=[0], colors='#6250d6', linewidths=2, linestyles='--')
ax[0].legend(handles=[Line2D([], [], marker='o', linestyle='', color=COLOR_CN, label='CN'),
                      Line2D([], [], marker='o', linestyle='', color=COLOR_DEM, label='Dementia'),
                      Line2D([], [], color='#0b0b0b', linewidth=2, label='Frontera SVM lineal'),
                      Line2D([], [], color='#6250d6', linewidth=2, linestyle='--', label='Frontera SVM RBF')],
             frameon=False, loc='lower left')
ax[0].set_title('SVM: frontera lineal y frontera RBF')

# Derecha: Random Forest (zona sombreada = predicción Dementia)
ax[1].contourf(xx, yy, (z_rf >= 0.5).astype(int), levels=[-0.5, 0.5, 1.5],
               cmap=ListedColormap(['#ffffff', '#fbe1d6']), alpha=0.7, zorder=0)
ax[1].contour(xx, yy, z_rf, levels=[0.5], colors=COLORES[2], linewidths=2)
ax[1].legend(handles=[Line2D([], [], marker='o', linestyle='', color=COLOR_CN, label='CN'),
                      Line2D([], [], marker='o', linestyle='', color=COLOR_DEM, label='Dementia'),
                      Line2D([], [], color=COLORES[2], linewidth=2, label='Frontera Random Forest')],
             frameon=False, loc='lower left')
ax[1].set_title('Random Forest: frontera por regiones')

plt.suptitle('Fronteras de decisión con las 2 variables más relevantes', fontweight='bold')
plt.tight_layout()
plt.show()

### Preguntas de análisis

1. ¿Qué modelo presentó mejor desempeño?<br>
El SVM RBF, que obtuvo la mejor Balanced Accuracy (0.873), F1 (0.832) y accuracy (0.923) de todos los modelos con valores por defecto, incluidos los de árboles. En ROC-AUC la Regresión Logística L2 quedó ligeramente arriba (0.947 contra 0.945) y SGD tuvo el mayor recall (0.806); sin embargo, las diferencias entre los modelos útiles son menores que la desviación entre folds.

2. ¿Los kernels no lineales mejoraron respecto al modelo lineal?<br>
No de forma relevante. El RBF quedó prácticamente igual que el SVM lineal (+0.005 de Balanced Accuracy y −0.002 de ROC-AUC) y el polinomial quedó por debajo.

3. ¿Los resultados sugieren que la relación entre variables es principalmente lineal o no lineal?<br>
Principalmente lineal. El SVM lineal y las regresiones logísticas obtienen casi el mismo resultado que el RBF, y el kernel polinomial, que fuerza una frontera más compleja, empeora. Además, los modelos de árboles, que captan relaciones no lineales e interacciones sin necesidad de un kernel, tampoco superaron a los modelos lineales.

4. ¿Existe algún modelo con desempeño considerablemente inferior?<br>
Sí, además del Dummy (que sirve de referencia), el SVM polinomial y AdaBoost con valores por defecto: ambos tienen la Balanced Accuracy más baja (0.841) y los ROC-AUC más bajos (0.933 y 0.936). El polinomial, además, tiene el recall más bajo (0.700) y la mayor variación entre folds.

5. ¿Qué kernel seleccionaría para la optimización y por qué?<br>
Se optimizarán los dos kernels no lineales. El RBF por ser el más prometedor empíricamente (mejor Balanced Accuracy y F1, y estable entre folds), y el polinomial para comprobar si su mal desempeño se debe a sus valores por defecto (degree = 3 y coef0 = 0) o al kernel en sí.

# Fase 6. Optimización de hiperparámetros

## 11. Selección de los modelos a optimizar

Se optimizan con `GridSearchCV` la Regresión Logística, los dos kernels no lineales del SVM (RBF y polinomial), Random Forest y los tres modelos de boosting (AdaBoost, Gradient Boosting y XGBoost), usando los mismos 5 folds y solo Train. Con 1,299 pacientes en Train, el espacio de búsqueda de cada modelo se puede explorar completo en un tiempo razonable. No se optimizan el SVM lineal ni SGD: ambos son clasificadores lineales, igual que la Regresión Logística, que ya representa a esa familia en la comparación final.

La métrica de optimización es la **Balanced Accuracy** (promedio del recall de cada clase), en lugar del ROC-AUC usado en la Práctica 1. El ROC-AUC mide qué tan bien se ordenan los pacientes, pero no el punto de corte: dos configuraciones con prácticamente el mismo ROC-AUC pueden detectar cantidades muy distintas de pacientes con demencia (ver la justificación en la sección 13). ROC-AUC, Recall y F1 se registran en cada combinación como métricas de apoyo.

Después de optimizar los modelos individuales se construyen los ensambles que combinan modelos de familias distintas (Voting y Stacking, secciones 12.9 y 12.10).

## 12. Hiperparámetros a explorar

- **C:** controla cuánto se castiga cada error o violación del margen. Con C pequeño el modelo tolera más errores, el margen es más ancho y la frontera más simple (más regularización, riesgo de underfitting). Con C grande el modelo intenta clasificar bien a casi todos los pacientes de Train, el margen se angosta y la frontera se vuelve más compleja (menos regularización, riesgo de overfitting).
- **γ (RBF):** controla el radio de influencia de cada vector de soporte. Con γ pequeño cada paciente influye en una zona amplia y la frontera es suave, casi lineal. Con γ grande la influencia es muy local y la frontera se adapta a pacientes individuales, lo que puede llevar a memorizar Train.
- **degree (polinomial):** grado del polinomio. Un grado mayor permite fronteras más curvas e interacciones de más variables, pero aumenta la complejidad, el riesgo de overfitting y el tiempo de entrenamiento.
- **coef0 (polinomial):** pondera los términos de menor grado frente a los de mayor grado.
- **class_weight:** con `"balanced"`, cada error se multiplica por un peso inversamente proporcional a la frecuencia de su clase, de modo que equivocarse con un paciente con demencia cuesta más. Se justifica por la distribución de Train (75% CN y 25% Dementia), calculada abajo.

**Random Forest:**

- **n_estimators:** número de árboles. Más árboles no producen overfitting: el promedio solo se vuelve más estable, hasta que deja de mejorar y solo aumenta el tiempo. Por eso se fija en 300 durante la búsqueda y su efecto se analiza aparte.
- **max_depth:** profundidad máxima de cada árbol. Árboles profundos captan interacciones complejas pero memorizan Train (overfitting); árboles poco profundos dan fronteras más simples (riesgo de underfitting). `None` deja crecer el árbol hasta que sus hojas son puras.
- **min_samples_leaf:** número mínimo de pacientes en cada hoja. Valores grandes impiden que el árbol cree hojas para uno o dos pacientes, suavizan la frontera y regularizan el modelo.
- **max_features:** proporción de variables que cada división puede considerar. Valores pequeños (`"sqrt"`, ≈ 3 de 11) hacen a los árboles más distintos entre sí y reducen la varianza del bosque; valores grandes (0.5) hacen árboles individuales más fuertes pero más parecidos.
- **class_weight:** igual que en los demás modelos, `"balanced"` hace que cada paciente con demencia pese ≈ 2 veces más al elegir las divisiones y al calcular las proporciones de cada hoja.

**Boosting (AdaBoost, Gradient Boosting y XGBoost):**

- **n_estimators:** número de árboles. A diferencia de Random Forest, en boosting **sí** puede producir overfitting: cada árbol nuevo sigue corrigiendo los errores de Train, aunque esos errores ya sean ruido. Por eso se optimiza junto con learning_rate.
- **learning_rate:** cuánto aporta cada árbol a la predicción. Con valores pequeños el modelo aprende más despacio y necesita más árboles, pero suele generalizar mejor; con valores grandes aprende rápido y se sobreajusta antes.
- **Profundidad de los árboles** (`estimator__max_depth` en AdaBoost, `max_depth` en Gradient Boosting y XGBoost): árboles de 1 nivel solo usan una variable; con 2 a 4 niveles pueden captar interacciones entre variables, con mayor riesgo de overfitting.
- **subsample:** proporción de pacientes de Train que usa cada árbol. Con valores menores que 1 (Gradient Boosting estocástico) cada árbol ve una muestra distinta, lo que agrega aleatoriedad y reduce el overfitting.
- **colsample_bytree (XGBoost):** proporción de variables que usa cada árbol; tiene el mismo objetivo que max_features en Random Forest.
- **Desbalance:** en AdaBoost se aplica `class_weight` al árbol base; en Gradient Boosting, que no tiene `class_weight`, se usan pesos por paciente (sección 12.7); en XGBoost se usa `scale_pos_weight`, el peso de la clase positiva, que con el valor CN / Dementia de Train (≈ 3.06) equivale a `class_weight="balanced"`.

In [ ]:
pesos = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
print('Pacientes en Train  -> CN:', (y_train == 0).sum(), '| Dementia:', (y_train == 1).sum())
print(f'Pesos con class_weight="balanced" -> CN: {pesos[0]:.2f} | Dementia: {pesos[1]:.2f}')
print(f'Un error en Dementia cuesta {pesos[1] / pesos[0]:.1f} veces más que un error en CN')

In [ ]:
def busqueda(modelo, parametros, nombre):
    grid = GridSearchCV(crear_pipeline(modelo), parametros, cv=cv, scoring=metricas,
                        refit='balanced_accuracy', n_jobs=-1, return_train_score=True)
    inicio = time.perf_counter()
    grid.fit(X_train_fe, y_train)
    i = grid.best_index_
    r = grid.cv_results_
    print(nombre)
    print(f'  Combinaciones: {len(r["params"])} | Tiempo: {time.perf_counter() - inicio:.1f} s')
    print('  Mejores hiperparámetros:', {nombre_param(k): formato(v) for k, v in grid.best_params_.items()})
    print(f'  Balanced Accuracy CV: {r["mean_test_balanced_accuracy"][i]:.4f} (desv. {r["std_test_balanced_accuracy"][i]:.4f})')
    print(f'  ROC-AUC CV: {r["mean_test_roc_auc"][i]:.4f}')
    print(f'  Recall CV:  {r["mean_test_recall"][i]:.4f} | F1 CV: {r["mean_test_f1"][i]:.4f}')
    print(f'  Brecha Bal. Acc. (Train − validación): {r["mean_train_balanced_accuracy"][i] - r["mean_test_balanced_accuracy"][i]:.4f}')
    return grid


def nombre_param(clave):
    # 'modelo' como hiperparámetro es la versión con o sin pesos de Gradient Boosting (sección 12.7)
    return 'balanceo' if clave == 'modelo' else clave.replace('modelo__', '')


def formato(v):
    # None puede llegar como NaN según la versión de pandas
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return 'None'
    if hasattr(v, 'fit'):
        return 'balanced' if type(v).__name__ == 'GradientBoostingBalanceado' else 'None'
    if isinstance(v, (int, float, np.integer, np.floating)) and float(v).is_integer():
        return str(int(v))
    return str(v)


def tabla_busqueda(grid, params):
    r = pd.DataFrame(grid.cv_results_)
    columnas = ['param_modelo' if p == 'balanceo' else 'param_modelo__' + p for p in params]
    t = r[columnas].apply(lambda col: col.map(formato))
    t.columns = params
    t['Bal. Acc. train'] = r['mean_train_balanced_accuracy']
    t['Bal. Acc. val'] = r['mean_test_balanced_accuracy']
    t['Desv val'] = r['std_test_balanced_accuracy']
    t['Recall val'] = r['mean_test_recall']
    t['ROC-AUC val'] = r['mean_test_roc_auc']
    t['F1 val'] = r['mean_test_f1']
    t['Brecha'] = t['Bal. Acc. train'] - t['Bal. Acc. val']
    t['Rank'] = r['rank_test_balanced_accuracy']
    return t.sort_values('Rank')

### 12.1 Regresión Logística (referencia)

In [ ]:
grid_log = busqueda(
    LogisticRegression(max_iter=5000),
    {'modelo__C': [0.001, 0.01, 0.1, 1, 10, 100],
     'modelo__class_weight': [None, 'balanced']},
    'Logística optimizada'
)

### 12.2 SVM RBF

In [ ]:
grid_rbf = busqueda(
    SVC(kernel='rbf'),
    {'modelo__C': [0.01, 0.1, 1, 10, 100],
     'modelo__gamma': ['scale', 0.001, 0.01, 0.1, 1, 10],
     'modelo__class_weight': [None, 'balanced']},
    'SVM RBF optimizado'
)

In [ ]:
res_rbf = tabla_busqueda(grid_rbf, ['C', 'gamma', 'class_weight'])
res_rbf.head(15).round(4)

In [ ]:
mejor_cw_rbf = formato(grid_rbf.best_params_['modelo__class_weight'])
orden_gamma = ['scale', '0.001', '0.01', '0.1', '1', '10']
orden_c = ['0.01', '0.1', '1', '10', '100']

calor = (res_rbf[res_rbf['class_weight'] == mejor_cw_rbf]
         .pivot_table(index='C', columns='gamma', values='Bal. Acc. val')
         .loc[orden_c, orden_gamma])

fig, ax = plt.subplots(figsize=(8, 4.8))
im = ax.imshow(calor.values, cmap='Blues', aspect='auto')
plt.colorbar(im, label='Balanced Accuracy validación')
ax.set_xticks(range(len(orden_gamma)))
ax.set_xticklabels(orden_gamma)
ax.set_yticks(range(len(orden_c)))
ax.set_yticklabels(orden_c)
ax.set_xlabel('gamma')
ax.set_ylabel('C')
ax.grid(False)
umbral = np.nanmin(calor.values) + (np.nanmax(calor.values) - np.nanmin(calor.values)) * 0.6
for i in range(calor.shape[0]):
    for j in range(calor.shape[1]):
        v = calor.values[i, j]
        ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=8,
                color='white' if v > umbral else '#0b0b0b')
ax.set_title(f'SVM RBF: Balanced Accuracy según C y gamma (class_weight = {mejor_cw_rbf})')
plt.tight_layout()
plt.show()

In [ ]:
mejor_gamma_rbf = formato(grid_rbf.best_params_['modelo__gamma'])
efecto_c = (res_rbf[(res_rbf['gamma'] == mejor_gamma_rbf) & (res_rbf['class_weight'] == mejor_cw_rbf)]
            .assign(C_num=lambda t: t['C'].astype(float)).sort_values('C_num'))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(efecto_c['C_num'], efecto_c['Bal. Acc. train'], marker='o', linewidth=2, color=COLORES[0], label='Train')
ax.plot(efecto_c['C_num'], efecto_c['Bal. Acc. val'], marker='o', linewidth=2, color=COLORES[1], label='Validación')
ax.set_xscale('log')
ax.set_xlabel('C (escala logarítmica)')
ax.set_ylabel('Balanced Accuracy')
ax.legend(frameon=False)
ax.set_title(f'SVM RBF: efecto de C (gamma = {mejor_gamma_rbf})')
plt.tight_layout()
plt.show()

### Análisis

1. Efecto de C sobre la regularización y la tolerancia a errores:<br>
Con γ = 0.1 y class_weight = "balanced", al aumentar C la Balanced Accuracy de Train sube siempre (de 0.882 con C = 0.01 hasta ≈ 1.00 con C = 100), mientras que la de validación sube solo hasta C = 1 (0.900) y después baja (0.876 con C = 10 y 0.841 con C = 100). Con C pequeño el modelo tolera muchos errores y hace underfitting; con C grande deja de tolerar errores, se ajusta a cada paciente de Train y hace overfitting. En el extremo, con C = 0.01 y γ = 0.001 el modelo es tan simple que predice una sola clase (Balanced Accuracy 0.500).

2. ¿Qué ocurre cuando γ aumenta o disminuye?<br>
Con γ grande el modelo se vuelve muy local y memoriza Train: con γ = 1 la Balanced Accuracy de validación cae a 0.50–0.60 y con γ = 10 a 0.500 para todos los valores de C. Con γ pequeño (0.001) la frontera es muy suave y necesita un C mayor para funcionar: con C ≤ 0.1 se queda entre 0.500 y 0.867, pero con C = 10 o 100 llega a 0.893–0.895. La zona buena está entre γ = 0.001 y 0.1, donde varias combinaciones quedan entre 0.89 y 0.90, es decir, el resultado es estable dentro de ese rango.

### 12.3 SVM polinomial

In [ ]:
grid_poly = busqueda(
    SVC(kernel='poly'),
    {'modelo__degree': [2, 3, 4, 5],
     'modelo__coef0': [0, 1, 5],
     'modelo__C': [0.1, 1, 10, 100],
     'modelo__class_weight': [None, 'balanced']},
    'SVM polinomial optimizado'
)

In [ ]:
res_poly = tabla_busqueda(grid_poly, ['degree', 'coef0', 'C', 'class_weight'])
res_poly.head(15).round(4)

In [ ]:
mejor_c_poly = formato(grid_poly.best_params_['modelo__C'])
mejor_cw_poly = formato(grid_poly.best_params_['modelo__class_weight'])
efecto_degree = res_poly[(res_poly['C'] == mejor_c_poly) & (res_poly['class_weight'] == mejor_cw_poly)]

fig, ax = plt.subplots(figsize=(7, 4))
for i, c0 in enumerate(['0', '1', '5']):
    d = efecto_degree[efecto_degree['coef0'] == c0].assign(g=lambda t: t['degree'].astype(int)).sort_values('g')
    ax.plot(d['g'], d['Bal. Acc. val'], marker='o', linewidth=2, color=COLORES[i], label=f'coef0 = {c0}')
ax.set_xticks([2, 3, 4, 5])
ax.set_xlabel('degree')
ax.set_ylabel('Balanced Accuracy validación')
ax.legend(frameon=False)
ax.set_title(f'SVM polinomial: efecto de degree (C = {mejor_c_poly})')
plt.tight_layout()
plt.show()

### Análisis

1. ¿Cómo influye degree en la complejidad del modelo?<br>
A mayor grado, el kernel combina más variables entre sí y la frontera puede ser más curva, lo que aumenta la complejidad. En los resultados, con C = 1 y class_weight = "balanced", la Balanced Accuracy de validación baja conforme sube el grado (con coef0 = 1: 0.901, 0.888, 0.871 y 0.863 para grados 2 a 5), mientras que la brecha entre Train y validación crece (0.013 con grado 2 contra 0.052 con grado 3 y C = 1). Con coef0 = 0 los resultados son irregulares (0.79 a 0.87) porque el kernel pierde los términos de menor grado. El mejor grado fue el más bajo (2), lo que confirma que la relación es casi lineal.

### 12.4 Random Forest

In [ ]:
grid_rf = busqueda(
    RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    {'modelo__max_depth': [None, 4, 8, 12],
     'modelo__min_samples_leaf': [1, 5, 10, 20],
     'modelo__max_features': ['sqrt', 0.5],
     'modelo__class_weight': [None, 'balanced']},
    'Random Forest optimizado'
)

In [ ]:
res_rf = tabla_busqueda(grid_rf, ['max_depth', 'min_samples_leaf', 'max_features', 'class_weight'])
res_rf.head(15).round(4)

In [ ]:
mejor_rf = res_rf.loc[grid_rf.best_index_]
mejor_mf_rf, mejor_cw_rf = mejor_rf['max_features'], mejor_rf['class_weight']
orden_depth = ['4', '8', '12', 'None']
orden_leaf = ['1', '5', '10', '20']
sub_rf = res_rf[(res_rf['max_features'] == mejor_mf_rf) & (res_rf['class_weight'] == mejor_cw_rf)]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
for eje, col, titulo, cmap in [(ax[0], 'Bal. Acc. val', 'Balanced Accuracy en validación', 'Blues'),
                               (ax[1], 'Brecha', 'Brecha Train − validación', 'Oranges')]:
    calor = sub_rf.pivot_table(index='max_depth', columns='min_samples_leaf', values=col).loc[orden_depth, orden_leaf]
    im = eje.imshow(calor.values, cmap=cmap, aspect='auto')
    plt.colorbar(im, ax=eje)
    eje.set_xticks(range(len(orden_leaf)))
    eje.set_xticklabels(orden_leaf)
    eje.set_yticks(range(len(orden_depth)))
    eje.set_yticklabels(orden_depth)
    eje.set_xlabel('min_samples_leaf')
    eje.set_ylabel('max_depth')
    eje.grid(False)
    for i in range(calor.shape[0]):
        for j in range(calor.shape[1]):
            v = calor.values[i, j]
            eje.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=9,
                     color='white' if v > np.nanpercentile(calor.values, 70) else '#0b0b0b')
    eje.set_title(titulo)
plt.suptitle(f'Random Forest (max_features = {mejor_mf_rf}, class_weight = {mejor_cw_rf})', fontweight='bold')
plt.tight_layout()
plt.show()

### Efecto del número de árboles

Con los mejores hiperparámetros encontrados se varía solo `n_estimators`, con los mismos 5 folds.

In [ ]:
n_arboles = [10, 25, 50, 100, 200, 300, 500]
grid_arboles = GridSearchCV(crear_pipeline(RandomForestClassifier(random_state=RANDOM_STATE)),
                            {**{k: [v] for k, v in grid_rf.best_params_.items()},
                             'modelo__n_estimators': n_arboles},
                            cv=cv, scoring=metricas, refit=False, n_jobs=-1)
grid_arboles.fit(X_train_fe, y_train)
r_arb = grid_arboles.cv_results_

fig, ax = plt.subplots(figsize=(7.5, 4))
for m, color in [('balanced_accuracy', COLORES[0]), ('roc_auc', COLORES[1])]:
    media, desv = r_arb['mean_test_' + m], r_arb['std_test_' + m]
    ax.plot(n_arboles, media, marker='o', linewidth=2, color=color, label=nombres_metricas[m])
    ax.fill_between(n_arboles, media - desv, media + desv, color=color, alpha=0.12)
ax.set_xscale('log')
ax.set_xticks(n_arboles)
ax.set_xticklabels(n_arboles)
ax.set_xlabel('n_estimators (escala logarítmica)')
ax.set_ylabel('Promedio en validación')
ax.legend(frameon=False)
ax.set_title('Efecto del número de árboles')
plt.tight_layout()
plt.show()

print(pd.DataFrame({'n_estimators': n_arboles,
                    'Bal. Acc. val': r_arb['mean_test_balanced_accuracy'],
                    'ROC-AUC val': r_arb['mean_test_roc_auc']}).set_index('n_estimators').round(4))

### Análisis

Con max_features = 0.5 y class_weight = "balanced", el hiperparámetro que más controló la complejidad fue min_samples_leaf. Con hojas de un solo paciente, la brecha es de 0.122 a 0.151 cuando max_depth ≥ 8 y la Balanced Accuracy de validación es la más baja del mapa (0.849 a 0.859), es decir, el bosque memoriza Train. Al subir min_samples_leaf la brecha baja de forma constante (con max_depth = 12: 0.150, 0.077, 0.050 y 0.034 para 1, 5, 10 y 20 pacientes por hoja), pero con 20 la validación vuelve a bajar (0.863), ya que los árboles se vuelven demasiado simples. max_depth actúa en el mismo sentido: con profundidad 4 la brecha es la menor (0.032 a 0.048), pero el modelo pierde capacidad (0.858 a 0.867), y con 12 y None los resultados son prácticamente idénticos (diferencia ≤ 0.001) debido a que los árboles casi nunca llegan a más de 12 niveles. El mejor punto fue max_depth = 8 con min_samples_leaf = 5 (0.879, brecha 0.070).

Las 15 mejores combinaciones usan class_weight = "balanced": igual que en la Regresión Logística y los SVM, fue el hiperparámetro con más efecto, ya que sin pesos el bosque favorece a la clase mayoritaria (CN). En cambio, max_features casi no influyó (0.879 con 0.5 y 0.877 con "sqrt"), una diferencia mucho menor que la desviación entre folds (≈ 0.016).

En cuanto al número de árboles, la Balanced Accuracy sube rápido de 10 árboles (0.854) a 100 (0.875) y después casi no cambia (0.877 con 200 y 0.879 con 300), mientras que el ROC-AUC se estabiliza desde 100 árboles (≈ 0.944). Con 500 árboles bajó a 0.872; no se trata de overfitting (agregar árboles no sobreajusta un Random Forest), sino de variación aleatoria, ya que con solo 320 pacientes con demencia en Train unos pocos pacientes cerca del corte cambian la métrica. Además, 300 fue el valor con el que se eligieron los demás hiperparámetros, lo que le da una pequeña ventaja. Por ende, a partir de 100 a 200 árboles el modelo deja de mejorar y solo aumenta el tiempo de entrenamiento.

### 12.5 Importancia de las variables

Random Forest permite medir qué variables usa más el modelo. Se calculan dos tipos de importancia con el mejor Random Forest:

- **Importancia por impureza (MDI):** cuánto reduce cada variable la impureza de Gini en todas las divisiones de todos los árboles. Se calcula con Train y puede sobrevalorar variables continuas con muchos valores distintos.
- **Importancia por permutación:** cuánto baja la Balanced Accuracy al revolver los valores de una variable. Se calcula en el fold de validación de cada uno de los 5 folds (pacientes que el modelo no vio), por lo que no usa Test.

In [ ]:
etiquetas_fe = {'Edad': 'Edad', 'Escolaridad': 'Escolaridad', 'APOE4': 'APOE4', 'Sexo': 'Sexo',
                'Hipocampo_total_ICV': 'Hipocampo / ICV', 'Entorrinal_total_ICV': 'Entorrinal / ICV',
                'Amigdala_total_ICV': 'Amígdala / ICV', 'Ventriculos_total_ICV': 'Ventrículos laterales / ICV',
                'Vent_inferior_total_ICV': 'Ventrículo lat. inferior / ICV', 'CortexVol_ICV': 'Corteza / ICV',
                'Grosor_medio': 'Grosor cortical medio'}

# Importancia por impureza (MDI), del modelo reentrenado con todo Train
rf_final = grid_rf.best_estimator_
nombres_rf = [n.split('__', 1)[1] for n in rf_final.named_steps['prep'].get_feature_names_out()]
nombres_rf = ['Sexo' if n.startswith('Sexo') else n for n in nombres_rf]
imp_mdi = pd.Series(rf_final.named_steps['modelo'].feature_importances_, index=nombres_rf)

# Importancia por permutación en el fold de validación de cada fold
perm_folds = []
for idx_tr, idx_val in cv.split(X_train_fe, y_train):
    modelo_fold = clone(rf_final).fit(X_train_fe.iloc[idx_tr], y_train.iloc[idx_tr])
    r = permutation_importance(modelo_fold, X_train_fe.iloc[idx_val], y_train.iloc[idx_val],
                               scoring='balanced_accuracy', n_repeats=10,
                               random_state=RANDOM_STATE, n_jobs=-1)
    perm_folds.append(pd.Series(r.importances_mean, index=X_train_fe.columns))
perm_folds = pd.DataFrame(perm_folds)

importancias = pd.DataFrame({'MDI': imp_mdi,
                             'Permutación (media)': perm_folds.mean(),
                             'Permutación (desv.)': perm_folds.std()})
importancias = importancias.sort_values('Permutación (media)', ascending=False)
importancias.index = importancias.index.map(etiquetas_fe)
importancias.round(4)

In [ ]:
orden_imp = importancias.sort_values('Permutación (media)').index
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
ax[0].barh(orden_imp, importancias.loc[orden_imp, 'MDI'], color=COLORES[0], height=0.6)
ax[0].set_xlabel('Reducción media de impureza (Gini)')
ax[0].set_title('Importancia por impureza (MDI)')
ax[1].barh(orden_imp, importancias.loc[orden_imp, 'Permutación (media)'],
           xerr=importancias.loc[orden_imp, 'Permutación (desv.)'], color=COLORES[1], height=0.6,
           error_kw={'elinewidth': 1, 'ecolor': COLOR_TEXTO, 'capsize': 3})
ax[1].axvline(0, color=COLOR_TEXTO, linewidth=1)
ax[1].set_xlabel('Caída de la Balanced Accuracy al permutar')
ax[1].set_title('Importancia por permutación (validación)')
for eje in ax:
    eje.grid(axis='y', visible=False)
plt.suptitle('¿Qué variables usa el Random Forest?', fontweight='bold')
plt.tight_layout()
guardar_figura(fig, 'importancia_random_forest')
plt.show()

### Análisis

Por permutación, las variables más importantes son el ventrículo lateral inferior / ICV (0.072), el hipocampo / ICV (0.048), la amígdala / ICV (0.038) y la corteza entorrinal / ICV (0.029), seguidas de APOE4 (0.024). Esto es coherente con el EDA, ya que son las variables con mayor correlación con el diagnóstico (ventrículos laterales inferiores ≈ 0.54 a 0.56, hipocampo ≈ −0.55 a −0.58 y amígdala ≈ −0.49 a −0.53), y corresponden al lóbulo temporal medial, la región que se atrofia primero en la enfermedad de Alzheimer (el ventrículo lateral inferior se agranda cuando el hipocampo vecino se reduce). Edad (0.002) y Sexo (0.0003) casi no aportan: la edad sí se correlaciona con el diagnóstico (0.21), pero su efecto ya está contenido en los volúmenes cerebrales, que también disminuyen con la edad.

La importancia por impureza (MDI) coincide con la de permutación en cuáles variables son importantes y cuáles no, pero no en el orden: MDI pone primero al hipocampo (0.308) y a la amígdala (0.211), y en tercer lugar al ventrículo lateral inferior (0.169). La diferencia se debe a que MDI se calcula con Train y reparte la importancia entre variables correlacionadas según cuál elige cada árbol para dividir (el hipocampo y la amígdala tienen una correlación de ≈ 0.78), mientras que la permutación mide cuánto empeora el modelo con pacientes que no vio, por lo que es la referencia más confiable. En ambas, Sexo, Edad, Escolaridad y los ventrículos laterales son las cuatro variables menos importantes.

### 12.6 AdaBoost

Se optimizan el número de árboles, learning_rate, la profundidad del árbol base (1 = *stump*, 2 o 3 niveles) y el `class_weight` del árbol base. Son 96 combinaciones.

In [ ]:
def mapa_calor(tabla, filtro, filas, columnas, titulo, archivo):
    # Balanced Accuracy de validación y brecha para dos hiperparámetros,
    # con los demás fijos en su mejor valor (filtro)
    sub = tabla
    for p, v in filtro.items():
        sub = sub[sub[p] == v]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    for eje, valor, cmapa, nombre in [(ax[0], 'Bal. Acc. val', 'Blues', 'Balanced Accuracy en validación'),
                                      (ax[1], 'Brecha', 'Oranges', 'Brecha Train − validación')]:
        calor = sub.pivot_table(index=filas, columns=columnas, values=valor)
        calor = calor.loc[sorted(calor.index, key=float), sorted(calor.columns, key=float)]
        im = eje.imshow(calor.values, cmap=cmapa, aspect='auto')
        eje.set_xticks(range(calor.shape[1]))
        eje.set_xticklabels(calor.columns)
        eje.set_yticks(range(calor.shape[0]))
        eje.set_yticklabels(calor.index)
        eje.set_xlabel(columnas)
        eje.set_ylabel(filas)
        eje.grid(False)
        limite = calor.values.min() + 0.6 * (calor.values.max() - calor.values.min())
        for a in range(calor.shape[0]):
            for b in range(calor.shape[1]):
                eje.text(b, a, f'{calor.values[a, b]:.3f}', ha='center', va='center', fontsize=9,
                         color='white' if calor.values[a, b] > limite else '#0b0b0b')
        eje.set_title(nombre)
        fig.colorbar(im, ax=eje)
    fig.suptitle(titulo, fontweight='bold')
    plt.tight_layout()
    guardar_figura(fig, archivo)
    plt.show()


grid_ada = busqueda(
    AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE), random_state=RANDOM_STATE),
    {'modelo__n_estimators': [50, 100, 200, 400],
     'modelo__learning_rate': [0.05, 0.1, 0.5, 1.0],
     'modelo__estimator__max_depth': [1, 2, 3],
     'modelo__estimator__class_weight': [None, 'balanced']},
    'AdaBoost optimizado'
)

In [ ]:
res_ada = tabla_busqueda(grid_ada, ['n_estimators', 'learning_rate', 'estimator__max_depth', 'estimator__class_weight'])
res_ada.head(10).round(4)

In [ ]:
mejor_ada = res_ada.loc[grid_ada.best_index_]
mapa_calor(res_ada,
           {'estimator__max_depth': mejor_ada['estimator__max_depth'],
            'estimator__class_weight': mejor_ada['estimator__class_weight']},
           'n_estimators', 'learning_rate',
           f"AdaBoost (árbol base de profundidad {mejor_ada['estimator__max_depth']}, "
           f"class_weight = {mejor_ada['estimator__class_weight']})",
           'adaboost_busqueda')

### Análisis

Con el árbol base de profundidad 2 y sin class_weight, la combinación de learning_rate y n_estimators tuvo un efecto importante. Con learning_rate de 0.05 y 0.1 el modelo aprende muy despacio y, aun con 400 árboles, se queda entre 0.819 y 0.852 de Balanced Accuracy (underfitting), mientras que con learning_rate = 1 la brecha sube hasta 0.135 con 200 y 400 árboles, debido a que cada árbol corrige de más los errores de Train. El mejor punto fue un valor intermedio: learning_rate = 0.5 con 200 árboles (0.877, brecha 0.043).

Un árbol base de 2 niveles fue mejor que un *stump*, aunque por poco: la mejor combinación con *stumps* obtuvo 0.872 (400 árboles y learning_rate = 1), y con árboles de 3 niveles la brecha sube a más de 0.12. Por último, a diferencia de los demás modelos, las 10 mejores combinaciones usan class_weight = None; sin embargo, el recall del mejor AdaBoost (0.794) es el más bajo de todos los modelos optimizados.

### 12.7 Gradient Boosting

`GradientBoostingClassifier` no tiene `class_weight`. Para darle el mismo trato que a los demás modelos se define `GradientBoostingBalanceado`, una versión que en cada entrenamiento calcula un peso para cada paciente con `compute_sample_weight("balanced")`: los pacientes con demencia pesan ≈ 2.03 y los CN ≈ 0.66, exactamente lo que hace `class_weight="balanced"` en los demás modelos. En la búsqueda, la versión original y la balanceada se comparan como si fueran un hiperparámetro más (`balanceo`). Son 72 combinaciones.

In [ ]:
class GradientBoostingBalanceado(GradientBoostingClassifier):
    """GradientBoostingClassifier que pondera a los pacientes con pesos balanceados en cada entrenamiento
    (lo mismo que hace class_weight="balanced" en los demás modelos)."""

    def fit(self, X, y, sample_weight=None):
        pesos = compute_sample_weight('balanced', y)
        return super().fit(X, y, sample_weight=pesos)


grid_gb = busqueda(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    {'modelo': [GradientBoostingClassifier(random_state=RANDOM_STATE),
                GradientBoostingBalanceado(random_state=RANDOM_STATE)],
     'modelo__n_estimators': [100, 200, 400],
     'modelo__learning_rate': [0.05, 0.1],
     'modelo__max_depth': [2, 3, 4],
     'modelo__subsample': [0.8, 1.0]},
    'Gradient Boosting optimizado'
)

In [ ]:
res_gb = tabla_busqueda(grid_gb, ['balanceo', 'n_estimators', 'learning_rate', 'max_depth', 'subsample'])
res_gb.head(10).round(4)

In [ ]:
mejor_gb = res_gb.loc[grid_gb.best_index_]
mapa_calor(res_gb,
           {'balanceo': mejor_gb['balanceo'], 'learning_rate': mejor_gb['learning_rate'],
            'subsample': mejor_gb['subsample']},
           'max_depth', 'n_estimators',
           f"Gradient Boosting (balanceo = {mejor_gb['balanceo']}, learning_rate = {mejor_gb['learning_rate']}, "
           f"subsample = {mejor_gb['subsample']})",
           'gradient_boosting_busqueda')

# Efecto del balanceo: mejor combinación con y sin pesos
res_gb.groupby('balanceo')[['Bal. Acc. val', 'Recall val', 'ROC-AUC val']].max().round(4)

### Análisis

Con learning_rate = 0.1, subsample = 0.8 y pesos balanceados, los árboles más simples fueron los mejores: con max_depth = 2 y 100 árboles se obtuvo la mejor Balanced Accuracy (0.889) y la menor brecha (0.053), y al aumentar la profundidad o el número de árboles la brecha sube a entre 0.091 y 0.127 sin mejorar la validación (0.873 a 0.883). Esto confirma que, a diferencia de Random Forest, en boosting más árboles sí pueden llevar al overfitting.

Ponderar a los pacientes (balanceo) fue el cambio con mayor efecto: sin balanceo, la mejor Balanced Accuracy de todas las configuraciones fue de 0.873 y el mejor recall de 0.781, contra 0.889 y 0.859 con balanceo, mientras que el mejor ROC-AUC prácticamente no cambió (0.949 contra 0.948). Es decir, el balanceo no mejora la capacidad del modelo para ordenar a los pacientes, sino que mueve el punto de corte para detectar más pacientes con demencia. Por su parte, subsample tuvo un efecto pequeño: con los mismos hiperparámetros, 0.8 obtuvo 0.889 y 1.0 obtuvo 0.884, una diferencia menor que la desviación entre folds (≈ 0.018).

### 12.8 XGBoost

Se optimizan n_estimators, learning_rate, max_depth, subsample, colsample_bytree y `scale_pos_weight` (1 = sin balancear, o la razón CN / Dementia de Train). Son 96 combinaciones.

In [ ]:
razon_clases = round((y_train == 0).sum() / (y_train == 1).sum(), 2)
print('scale_pos_weight balanceado (CN / Dementia en Train):', razon_clases)

grid_xgb = busqueda(
    XGBClassifier(random_state=RANDOM_STATE, n_jobs=1, eval_metric='logloss'),
    {'modelo__n_estimators': [100, 300],
     'modelo__learning_rate': [0.05, 0.1],
     'modelo__max_depth': [2, 3, 4],
     'modelo__subsample': [0.8, 1.0],
     'modelo__colsample_bytree': [0.8, 1.0],
     'modelo__scale_pos_weight': [1, razon_clases]},
    'XGBoost optimizado'
)

In [ ]:
res_xgb = tabla_busqueda(grid_xgb, ['n_estimators', 'learning_rate', 'max_depth', 'subsample',
                                    'colsample_bytree', 'scale_pos_weight'])
res_xgb.head(10).round(4)

In [ ]:
mejor_xgb = res_xgb.loc[grid_xgb.best_index_]
mapa_calor(res_xgb,
           {'learning_rate': mejor_xgb['learning_rate'], 'subsample': mejor_xgb['subsample'],
            'colsample_bytree': mejor_xgb['colsample_bytree'], 'scale_pos_weight': mejor_xgb['scale_pos_weight']},
           'max_depth', 'n_estimators',
           f"XGBoost (learning_rate = {mejor_xgb['learning_rate']}, subsample = {mejor_xgb['subsample']}, "
           f"colsample_bytree = {mejor_xgb['colsample_bytree']}, scale_pos_weight = {mejor_xgb['scale_pos_weight']})",
           'xgboost_busqueda')

### Análisis

La optimización redujo la brecha de XGBoost, aunque sigue siendo una de las más altas: pasó de 0.137 con los valores por defecto a 0.104, y la Balanced Accuracy de validación subió de 0.863 a 0.888. Esto se debe principalmente a árboles menos profundos (4 en lugar de 6), a un learning_rate menor (0.1 en lugar de 0.3) y al submuestreo de pacientes y de variables (0.8); sin embargo, el modelo todavía llega a 0.991 en Train. Igual que en los demás modelos, el desbalance fue el factor con más efecto, ya que las 10 mejores combinaciones usan scale_pos_weight = 3.06.

Entre los tres modelos de boosting, Gradient Boosting balanceado obtuvo el mejor resultado (0.889), prácticamente empatado con XGBoost (0.888) y por encima de AdaBoost (0.878). La diferencia entre Gradient Boosting y XGBoost es mucho menor que la desviación entre folds, por lo que ambos funcionan de forma similar en este problema, mientras que AdaBoost queda atrás principalmente por su recall más bajo (0.794).

### Ensambles que combinan modelos de familias distintas

Random Forest y los modelos de boosting combinan muchos árboles del mismo tipo. Voting y Stacking combinan modelos **distintos**, que se equivocan de forma diferente, con la idea de que los errores de uno los compensen los otros.

Se usa el mejor modelo optimizado de cada familia, elegido por su Balanced Accuracy en validación cruzada:

- **Lineal:** Regresión Logística optimizada.
- **SVM:** el mejor entre el SVM RBF y el SVM polinomial optimizados.
- **Árboles:** el mejor entre Random Forest, AdaBoost, Gradient Boosting y XGBoost optimizados.

Se usan tres modelos (un número impar) para que el voto duro nunca termine empatado.

### 12.9 Voting

- **Voto duro:** cada modelo vota por una clase y gana la mayoría (2 de 3).
- **Voto suave:** se promedian las probabilidades de Dementia de los tres modelos y se clasifica como Dementia si el promedio es mayor que 0.5. Necesita que todos los modelos den probabilidades, por lo que el SVM se configura con `probability=True`.

In [ ]:
def mejor_de(grids):
    # Modelo con mayor Balanced Accuracy en validación cruzada
    return max(grids, key=lambda n: grids[n].best_score_)


svm_opt = {'SVM RBF optimizado': grid_rbf, 'SVM polinomial optimizado': grid_poly}
arboles_opt = {'Random Forest optimizado': grid_rf, 'AdaBoost optimizado': grid_ada,
               'Gradient Boosting optimizado': grid_gb, 'XGBoost optimizado': grid_xgb}
nombre_svm = mejor_de(svm_opt)
nombre_arboles = mejor_de(arboles_opt)

base_logistica = clone(grid_log.best_estimator_)
base_svm = clone(svm_opt[nombre_svm].best_estimator_)
base_svm_prob = clone(base_svm).set_params(modelo__probability=True, modelo__random_state=RANDOM_STATE)
base_arboles = clone(arboles_opt[nombre_arboles].best_estimator_)

print('Modelos base: Logística optimizada |', nombre_svm, '|', nombre_arboles, '\n')

voting_duro = VotingClassifier([('logistica', base_logistica), ('svm', base_svm), ('arboles', base_arboles)],
                               voting='hard')
voting_suave = VotingClassifier([('logistica', base_logistica), ('svm', base_svm_prob), ('arboles', base_arboles)],
                                voting='soft')

evaluar_cv(voting_duro, 'Voting duro', envolver=False, scoring=metricas_sin_auc)
evaluar_cv(voting_suave, 'Voting suave', envolver=False)

### 12.10 Stacking

En lugar de un voto fijo, Stacking entrena un **metamodelo** que aprende cuánto confiar en cada modelo base. Se usan los mismos tres modelos base y, como metamodelo, una Regresión Logística con `class_weight="balanced"`.

In [ ]:
stacking = StackingClassifier(
    estimators=[('logistica', base_logistica), ('svm', base_svm), ('arboles', base_arboles)],
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=5000),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    stack_method='auto'
)
evaluar_cv(stacking, 'Stacking', envolver=False)

# Coeficientes del metamodelo entrenado con todo Train
stacking_train = clone(stacking).fit(X_train_fe, y_train)
pd.DataFrame({'Modelo base': ['Logística optimizada', nombre_svm, nombre_arboles],
              'Puntaje que aporta': stacking_train.stack_method_,
              'Coeficiente del metamodelo': stacking_train.final_estimator_.coef_[0].round(3)})

### Análisis

Los modelos base de los ensambles son la Regresión Logística optimizada, el SVM polinomial optimizado y Gradient Boosting optimizado (el mejor de cada familia). Ningún ensamble superó de forma clara al mejor de sus modelos base en la validación cruzada de 5 folds: el voto duro obtuvo 0.900 y el Stacking 0.897, contra 0.901 del SVM polinomial. Sin embargo, ambos tienen un recall mayor (0.878 contra 0.859) y una menor desviación entre folds (0.016 y 0.019 contra 0.023).

El voto suave fue el más bajo (0.888), debido a que detecta menos pacientes con demencia (recall de 0.828), aunque su ROC-AUC (0.949) es mayor que el de cualquier modelo individual. Por su parte, el metamodelo del Stacking asigna el mayor coeficiente a la Regresión Logística (1.922), seguida de Gradient Boosting (1.035) y del SVM (0.933); sin embargo, estos coeficientes no son directamente comparables, ya que el SVM aporta su distancia a la frontera y los otros dos su probabilidad.

# Fase 7. Selección del modelo final

## 13. Mejor configuración de cada modelo

Los candidatos son los siete modelos optimizados y los tres ensambles mixtos (Voting duro, Voting suave y Stacking). La tabla resume su desempeño en la validación cruzada de 5 folds con la que se optimizaron.

In [ ]:
optimizados = {'Logística optimizada': grid_log, 'SVM RBF optimizado': grid_rbf,
               'SVM polinomial optimizado': grid_poly, 'Random Forest optimizado': grid_rf,
               'AdaBoost optimizado': grid_ada, 'Gradient Boosting optimizado': grid_gb,
               'XGBoost optimizado': grid_xgb}
ensambles_mixtos = {'Voting duro': voting_duro, 'Voting suave': voting_suave, 'Stacking': stacking}

candidatos = {n: clone(g.best_estimator_) for n, g in optimizados.items()}
candidatos.update({n: clone(m) for n, m in ensambles_mixtos.items()})


def medias_cv(nombre):
    # Promedio de cada métrica en los 5 folds (de la búsqueda o de evaluar_cv)
    if nombre in optimizados:
        g = optimizados[nombre]
        i = g.best_index_
        medias = {m: g.cv_results_['mean_test_' + m][i] for m in metricas}
        medias['desv_ba'] = g.cv_results_['std_test_balanced_accuracy'][i]
    else:
        s = resultados_cv[nombre]
        medias = {m: s['test_' + m].mean() if 'test_' + m in s else np.nan for m in metricas}
        medias['desv_ba'] = s['test_balanced_accuracy'].std()
    return medias


filas = []
for nombre in candidatos:
    m = medias_cv(nombre)
    if nombre in optimizados:
        detalle = {nombre_param(k): formato(v) for k, v in optimizados[nombre].best_params_.items()}
    else:
        detalle = f'Base: Logística optimizada, {nombre_svm}, {nombre_arboles}'
    filas.append({'Modelo': nombre, 'Configuración': detalle,
                  'Bal. Acc. CV': f"{m['balanced_accuracy']:.4f} ± {m['desv_ba']:.4f}",
                  'ROC-AUC CV': round(m['roc_auc'], 4), 'Recall CV': round(m['recall'], 4),
                  'F1 CV': round(m['f1'], 4), '_orden': m['balanced_accuracy']})
tabla_candidatos = pd.DataFrame(filas).sort_values('_orden', ascending=False).drop(columns='_orden').set_index('Modelo')
tabla_candidatos

### 13.1 Selección con validación cruzada repetida

La tabla anterior no basta para elegir, por dos razones:

1. Las diferencias entre los mejores candidatos suelen ser menores que la desviación entre folds (≈ 0.02).
2. Los hiperparámetros se eligieron con esos mismos 5 folds. Un modelo con 96 combinaciones probadas tiene más oportunidades de "acomodarse" a esos folds que la logística, con 12, por lo que su resultado en la tabla es más optimista.

Por eso la selección se hace con **validación cruzada repetida**: 5 folds × 5 repeticiones (25 evaluaciones por modelo) con particiones nuevas (`random_state=0`, distinto del 42 usado para optimizar). Todos los candidatos se evalúan con exactamente las mismas 25 particiones, de modo que también se puede contar en cuántas de ellas cada modelo supera a la Regresión Logística optimizada, el modelo más simple.

**Criterio de selección (definido antes de ver los resultados):** el candidato con mayor Balanced Accuracy promedio en las 25 evaluaciones.

In [ ]:
cv_repetida = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=0)

resultados_rep = {}
for nombre, modelo in candidatos.items():
    inicio = time.perf_counter()
    s = cross_validate(modelo, X_train_fe, y_train, cv=cv_repetida,
                       scoring=['balanced_accuracy', 'recall'], n_jobs=-1)
    resultados_rep[nombre] = s
    print(f"{nombre:<30} Bal. Acc. {s['test_balanced_accuracy'].mean():.4f} ± {s['test_balanced_accuracy'].std():.4f}"
          f" | Recall {s['test_recall'].mean():.4f} | {time.perf_counter() - inicio:.0f} s")

ba_logistica = resultados_rep['Logística optimizada']['test_balanced_accuracy']
filas = []
for nombre, s in resultados_rep.items():
    ba = s['test_balanced_accuracy']
    filas.append({'Modelo': nombre,
                  'Bal. Acc. (media)': ba.mean(), 'Bal. Acc. (desv.)': ba.std(),
                  'Recall (media)': s['test_recall'].mean(),
                  'Diferencia vs logística': (ba - ba_logistica).mean(),
                  'Particiones en que supera a la logística (de 25)': int((ba > ba_logistica).sum())})
tabla_repetida = pd.DataFrame(filas).set_index('Modelo').sort_values('Bal. Acc. (media)', ascending=False)

nombre_mejor = tabla_repetida.index[0]
best_model = clone(candidatos[nombre_mejor]).fit(X_train_fe, y_train)
print('\nModelo seleccionado:', nombre_mejor)
tabla_repetida.round(4)

In [ ]:
orden_rep = tabla_repetida.index
fig, ax = plt.subplots(figsize=(13, 4.8))
caja = ax.boxplot([resultados_rep[n]['test_balanced_accuracy'] for n in orden_rep],
                  patch_artist=True, widths=0.55, medianprops={'color': COLOR_TEXTO, 'linewidth': 1.5})
for parche, nombre in zip(caja['boxes'], orden_rep):
    parche.set_facecolor('#fbd3c2' if nombre == nombre_mejor else '#cfe0f5')
    parche.set_edgecolor(COLOR_DEM if nombre == nombre_mejor else COLOR_CN)
ax.set_xticks(range(1, len(orden_rep) + 1))
ax.set_xticklabels(orden_rep, rotation=25, ha='right')
ax.set_ylabel('Balanced Accuracy por partición')
ax.grid(axis='x', visible=False)
ax.set_title('Validación cruzada repetida (5 folds × 5 repeticiones)')
plt.tight_layout()
guardar_figura(fig, 'cv_repetida_candidatos')
plt.show()

### Reporte de la mejor configuración

1. Modelo seleccionado:<br>
Stacking, que combina la Regresión Logística optimizada, el SVM polinomial optimizado y Gradient Boosting optimizado, con una Regresión Logística como metamodelo.

2. Mejores hiperparámetros:<br>
Modelos base: Regresión Logística con C = 0.01 y class_weight = "balanced"; SVM polinomial con degree = 2, coef0 = 1, C = 1 y class_weight = "balanced"; Gradient Boosting balanceado con learning_rate = 0.1, max_depth = 2, n_estimators = 100 y subsample = 0.8. Metamodelo: Regresión Logística con class_weight = "balanced", entrenada con las predicciones *out-of-fold* de una validación cruzada interna de 5 folds.

3. Métrica utilizada durante la optimización y la selección:<br>
Balanced Accuracy, en la validación cruzada de 5 folds para optimizar los hiperparámetros y en la validación cruzada repetida (5 × 5) para elegir entre los candidatos.

4. Resultado en la validación cruzada repetida y comparación con la Regresión Logística optimizada:<br>
El Stacking obtuvo la mayor Balanced Accuracy promedio (0.8922 ± 0.0246, con un recall de 0.8656), seguido del SVM polinomial optimizado (0.8916) y del SVM RBF optimizado (0.8903). Frente a la Regresión Logística optimizada (0.8888 ± 0.0205) la diferencia es de solo +0.0033 y la supera en 16 de las 25 particiones, por lo que la ventaja es pequeña y menor que la desviación entre particiones. Además, todos los modelos de árboles quedaron por debajo de la logística (de −0.010 con XGBoost a −0.026 con AdaBoost).

5. Justificación de la métrica:<br>
La Balanced Accuracy es el promedio del recall de cada clase: (recall de Dementia + recall de CN) / 2. No se infla con la clase mayoritaria (un modelo que siempre dice CN tendría 75% de accuracy, pero 0.50 de Balanced Accuracy) y castiga los falsos negativos aunque Dementia sea solo el 25% de los pacientes. En la Práctica 1 se optimizó con ROC-AUC, pero el ROC-AUC mide qué tan bien se ordenan los pacientes y no depende del punto de corte. En la búsqueda del SVM RBF optimizada por ROC-AUC, la mejor configuración (C = 1, γ = 0.01, sin class_weight) tenía un recall de 0.744 en CV y dejaba sin detectar a 82 de los 320 pacientes con demencia de Train (predicciones OOF), mientras que la misma configuración con class_weight="balanced" tenía prácticamente el mismo ROC-AUC (0.9497 contra 0.9506) y un recall de 0.875. Como en este problema no detectar a un paciente con demencia es el error más grave, se eligió una métrica que sí refleja esa diferencia. El ROC-AUC se sigue reportando como métrica de apoyo.

## 14. Evaluación Out-of-Fold

Con `cross_val_predict`, cada paciente de Train se predice con un modelo que no lo vio durante su entrenamiento. Así se analizan los errores sin usar Test.

In [ ]:
def calcular_metricas(y_real, y_pred, puntaje):
    return {'Accuracy': accuracy_score(y_real, y_pred),
            'Balanced Accuracy': balanced_accuracy_score(y_real, y_pred),
            'Precision': precision_score(y_real, y_pred, zero_division=0),
            'Recall': recall_score(y_real, y_pred),
            'F1': f1_score(y_real, y_pred, zero_division=0),
            'ROC-AUC': roc_auc_score(y_real, puntaje)}


def graficar_matriz(ax, y_real, y_pred, titulo):
    cm = confusion_matrix(y_real, y_pred)
    porcentaje = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(porcentaje, cmap='Blues', vmin=0, vmax=1)
    etiquetas = ['CN', 'Dementia']
    ax.set_xticks([0, 1])
    ax.set_xticklabels(etiquetas)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(etiquetas)
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Diagnóstico real')
    ax.grid(False)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i, j]}\n({porcentaje[i, j]:.1%})', ha='center', va='center',
                    fontsize=11, color='white' if porcentaje[i, j] > 0.6 else '#0b0b0b')
    ax.set_title(titulo)


def tipo_puntaje(modelo):
    if isinstance(modelo, VotingClassifier) and modelo.voting == 'hard':
        return 'votos'
    return 'decision' if hasattr(modelo, 'decision_function') else 'probabilidad'


def obtener_puntaje(modelo, X):
    tipo = tipo_puntaje(modelo)
    # Voto duro: proporción de los modelos base que votan por Dementia
    if tipo == 'votos':
        return np.mean([est.predict(X) for est in modelo.estimators_], axis=0)
    # Logística, SVM, SGD, AdaBoost, Gradient Boosting y Stacking: distancia a la frontera
    if tipo == 'decision':
        return modelo.decision_function(X)
    # Random Forest, XGBoost y voto suave: probabilidad de Dementia
    return modelo.predict_proba(X)[:, 1]


def umbral_por_defecto(modelo):
    # Punto de corte con el que predict() decide la clase
    return 0.0 if tipo_puntaje(modelo) == 'decision' else 0.5


def calcular_puntaje_oof(modelo):
    # Cada paciente de Train se califica con un modelo que no lo vio (mismos 5 folds)
    puntaje = np.zeros(len(y_train))
    for idx_tr, idx_val in cv.split(X_train_fe, y_train):
        m = clone(modelo).fit(X_train_fe.iloc[idx_tr], y_train.iloc[idx_tr])
        puntaje[idx_val] = obtener_puntaje(m, X_train_fe.iloc[idx_val])
    return puntaje


umbral = umbral_por_defecto(best_model)
puntaje_oof = calcular_puntaje_oof(best_model)
y_oof = (puntaje_oof > umbral).astype(int)

metricas_oof = calcular_metricas(y_train, y_oof, puntaje_oof)
print('Métricas Out-of-Fold en Train —', nombre_mejor, f'(umbral por defecto = {umbral})')
for m, v in metricas_oof.items():
    print(f'  {m:<18}: {v:.4f}')

tn, fp, fn, tp = confusion_matrix(y_train, y_oof).ravel()
print(f'\nTN (CN bien clasificados):              {tn}')
print(f'FP (CN clasificados como Dementia):     {fp}')
print(f'FN (Dementia clasificados como CN):     {fn}')
print(f'TP (Dementia bien clasificados):        {tp}')

In [ ]:
etiquetas_puntaje = {'decision': 'Distancia a la frontera (decision_function)',
                     'probabilidad': 'Probabilidad de Dementia (predict_proba)',
                     'votos': 'Proporción de modelos que votan por Dementia'}

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={'width_ratios': [1, 1.4]})
graficar_matriz(ax[0], y_train, y_oof, 'Matriz de confusión (OOF en Train)')

bins = np.linspace(puntaje_oof.min(), puntaje_oof.max(), 40)
ax[1].hist(puntaje_oof[y_train.values == 0], bins=bins, alpha=0.6, color=COLOR_CN, label='CN')
ax[1].hist(puntaje_oof[y_train.values == 1], bins=bins, alpha=0.6, color=COLOR_DEM, label='Dementia')
ax[1].axvline(umbral, color=COLOR_TEXTO, linestyle='--', linewidth=1.5, label='Umbral por defecto')
ax[1].set_xlabel(etiquetas_puntaje[tipo_puntaje(best_model)])
ax[1].set_ylabel('Número de pacientes')
ax[1].legend(frameon=False)
ax[1].set_title('Separación de clases (OOF)')
plt.tight_layout()
plt.show()

### Análisis

1. Análisis de TN, FP, FN y TP:<br>
De los 979 pacientes CN de Train, 896 se clasificaron correctamente (TN, 91.5%) y 83 se marcaron como demencia (FP, 8.5%). De los 320 pacientes con demencia, 281 se detectaron (TP, 87.8%) y 39 se clasificaron como sanos (FN, 12.2%). En número, el modelo comete más errores con la clase CN (83 contra 39); sin embargo, en proporción se equivoca más con los pacientes con demencia (12.2% contra 8.5%), ya que son la clase minoritaria. Gracias a los pesos balanceados, ambos porcentajes de error son parecidos, en lugar de que el modelo favorezca a la clase CN.

2. Significado de los falsos positivos y falsos negativos en el contexto del proyecto:<br>
Un **falso positivo** es una persona cognitivamente normal que el modelo clasifica como demencia; en la práctica implicaría estudios adicionales, preocupación innecesaria para el paciente y su familia, y uso de recursos clínicos. Un **falso negativo** es un paciente con demencia que el modelo clasifica como sano; es el error más grave, porque retrasaría el diagnóstico, el tratamiento y la planeación del cuidado. Por eso se optimizó con Balanced Accuracy, que da el mismo peso al recall de Dementia que al de CN aunque Dementia sea la clase minoritaria.

### 14.1 Ajuste del umbral de decisión

Cada modelo decide la clase comparando su puntaje contra un umbral por defecto (0 para la distancia a la frontera y 0.5 para las probabilidades). En la práctica con la base de diabetes, mover el umbral cambió el recall de 0.17 a 0.80, porque Gradient Boosting no compensaba el desbalance. Aquí los modelos ya usan `class_weight="balanced"` (o su equivalente), por lo que se espera una ganancia pequeña.

El umbral se busca con los puntajes **OOF de Train**, nunca con Test, y se elige el que maximiza la Balanced Accuracy. **Regla de decisión:** el umbral nuevo solo se adopta si mejora la Balanced Accuracy OOF en al menos 0.01. Un umbral elegido con los mismos 1,299 pacientes tiende a verse mejor de lo que es, así que una mejora menor se considera ruido y se conserva el umbral por defecto.

In [ ]:
umbrales = np.unique(np.quantile(puntaje_oof, np.linspace(0.01, 0.99, 197)))
ba_umbrales = np.array([balanced_accuracy_score(y_train, (puntaje_oof > u).astype(int)) for u in umbrales])
recall_umbrales = np.array([recall_score(y_train, (puntaje_oof > u).astype(int)) for u in umbrales])
espec_umbrales = np.array([recall_score(y_train, (puntaje_oof > u).astype(int), pos_label=0) for u in umbrales])

umbral_optimo = umbrales[ba_umbrales.argmax()]
ganancia = ba_umbrales.max() - metricas_oof['Balanced Accuracy']
umbral_final = umbral_optimo if ganancia >= 0.01 else umbral

print(f'Umbral por defecto: {umbral:.4f} -> Balanced Accuracy OOF {metricas_oof["Balanced Accuracy"]:.4f}')
print(f'Umbral óptimo en OOF: {umbral_optimo:.4f} -> Balanced Accuracy OOF {ba_umbrales.max():.4f} (ganancia {ganancia:+.4f})')
print('Umbral adoptado:', round(umbral_final, 4),
      '(nuevo)' if umbral_final != umbral else '(se conserva el umbral por defecto: la ganancia es menor que 0.01)')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(umbrales, ba_umbrales, color=COLORES[0], linewidth=2, label='Balanced Accuracy')
ax.plot(umbrales, recall_umbrales, color=COLORES[1], linewidth=1.5, label='Recall (Dementia)')
ax.plot(umbrales, espec_umbrales, color=COLORES[2], linewidth=1.5, label='Especificidad (CN)')
ax.axvline(umbral, color=COLOR_TEXTO, linestyle='--', linewidth=1.2, label='Umbral por defecto')
ax.axvline(umbral_optimo, color=COLOR_DEM, linestyle=':', linewidth=1.5, label='Umbral óptimo (OOF)')
ax.set_xlabel(etiquetas_puntaje[tipo_puntaje(best_model)])
ax.set_ylabel('Valor (OOF en Train)')
ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=5, fontsize=8)
ax.set_title(f'Efecto del umbral — {nombre_mejor}')
plt.tight_layout()
guardar_figura(fig, 'ajuste_umbral')
plt.show()

### Análisis

El umbral que maximiza la Balanced Accuracy en las predicciones OOF fue 0.072, muy cercano al umbral por defecto (0), y solo mejora la Balanced Accuracy en +0.0015 (0.8982 contra 0.8967). Como la ganancia es menor que 0.01, se conserva el umbral por defecto. Esto es esperable debido a que el metamodelo ya usa class_weight = "balanced", por lo que su frontera ya está colocada en un punto donde el recall y la especificidad están equilibrados.

En la gráfica se observa el compromiso entre ambas métricas: al bajar el umbral aumenta el recall (se detectan más pacientes con demencia), pero baja la especificidad (más personas CN se clasifican como demencia), y al subirlo ocurre lo contrario. La Balanced Accuracy se mantiene prácticamente estable cerca del 0, donde ambas curvas se cruzan.

# V. Evaluación final

# Fase 8. Evaluación final sobre Test

## 15. Evaluación del modelo seleccionado

Test se usa por primera vez, con el modelo seleccionado entrenado con todo Train y con el umbral elegido en la sección 14.1.

In [ ]:
puntaje_test = obtener_puntaje(best_model, X_test_fe)
y_pred = (puntaje_test > umbral_final).astype(int)
metricas_test = calcular_metricas(y_test, y_pred, puntaje_test)

# OOF con el mismo umbral, para comparar en igualdad de condiciones
y_oof_final = (puntaje_oof > umbral_final).astype(int)
metricas_oof_final = calcular_metricas(y_train, y_oof_final, puntaje_oof)

comparacion_oof_test = pd.DataFrame({'OOF (Train)': metricas_oof_final, 'Test': metricas_test})
comparacion_oof_test['Diferencia'] = comparacion_oof_test['Test'] - comparacion_oof_test['OOF (Train)']
print('Modelo:', nombre_mejor, '| Umbral:', round(umbral_final, 4))
display(comparacion_oof_test.round(4))

print(classification_report(y_test, y_pred, target_names=['CN', 'Dementia']))

tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, y_pred).ravel()
print(f'TN: {tn_t} | FP: {fp_t} | FN: {fn_t} | TP: {tp_t}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
graficar_matriz(ax[0], y_train, y_oof_final, 'OOF en Train')
graficar_matriz(ax[1], y_test, y_pred, 'Test')
plt.suptitle(f'Matrices de confusión — {nombre_mejor}', fontweight='bold')
plt.tight_layout()
guardar_figura(fig, 'matrices_confusion_final')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
for y_real, puntaje, nombre, color in [(y_train, puntaje_oof, 'OOF en Train', COLORES[0]),
                                       (y_test, puntaje_test, 'Test', COLORES[1])]:
    fpr, tpr, _ = roc_curve(y_real, puntaje)
    ax.plot(fpr, tpr, linewidth=2, color=color,
            label=f'{nombre} (AUC = {roc_auc_score(y_real, puntaje):.3f})')
ax.plot([0, 1], [0, 1], linestyle='--', color='#8a8984', linewidth=1, label='Azar (AUC = 0.5)')
ax.set_xlabel('Tasa de falsos positivos')
ax.set_ylabel('Tasa de verdaderos positivos (Recall)')
ax.legend(frameon=False, loc='lower right')
ax.set_title(f'Curva ROC — {nombre_mejor}')
plt.tight_layout()
guardar_figura(fig, 'curva_roc_final')
plt.show()

### Curva Precision-Recall

Muestra la precisión (proporción de pacientes clasificados como Dementia que realmente tienen demencia) para cada valor de recall. A diferencia de la curva ROC, no usa los verdaderos negativos, por lo que es más sensible al desbalance de clases: un clasificador al azar tendría una precisión igual a la proporción de pacientes con demencia (≈ 0.25). Se resume con la precisión promedio (AP, *average precision*).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
for y_real, puntaje, nombre, color in [(y_train, puntaje_oof, 'OOF en Train', COLORES[0]),
                                       (y_test, puntaje_test, 'Test', COLORES[1])]:
    precision, recall, _ = precision_recall_curve(y_real, puntaje)
    ap = average_precision_score(y_real, puntaje)
    ax.plot(recall, precision, linewidth=2, color=color, label=f'{nombre} (AP = {ap:.3f})')
    print(f'Precisión promedio (AP) — {nombre}: {ap:.4f}')

# Punto de operación con el umbral adoptado, en Test
ax.scatter(recall_score(y_test, y_pred), precision_score(y_test, y_pred), s=60, zorder=5,
           color=COLOR_DEM, edgecolor='white', label='Umbral adoptado (Test)')
# Un clasificador al azar tiene una precisión igual a la proporción de Dementia
ax.axhline(y_test.mean(), linestyle='--', color='#8a8984', linewidth=1,
           label=f'Azar (prevalencia = {y_test.mean():.3f})')
ax.set_xlabel('Recall (sensibilidad)')
ax.set_ylabel('Precisión')
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, loc='lower left')
ax.set_title(f'Curva Precision-Recall — {nombre_mejor}')
plt.tight_layout()
guardar_figura(fig, 'curva_pr_final')
plt.show()

### 15.1 Intervalos de confianza (bootstrap)

Test tiene 80 pacientes con demencia: cada uno mueve el recall 1.25 puntos porcentuales, así que un solo número no dice qué tan estable es el resultado. Con *bootstrap* se toman 2,000 muestras de 325 pacientes de Test con reemplazo, se recalculan las métricas en cada una y el intervalo de confianza del 95% son los percentiles 2.5 y 97.5.

También se compara el modelo final contra la Regresión Logística optimizada en las mismas muestras (comparación pareada). Si el intervalo de la diferencia de Balanced Accuracy incluye el 0, la ventaja del modelo final sobre el modelo más simple no es concluyente con este tamaño de Test.

In [ ]:
modelo_ref = clone(grid_log.best_estimator_).fit(X_train_fe, y_train)
pred_ref = modelo_ref.predict(X_test_fe)

rng = np.random.default_rng(RANDOM_STATE)
yt = y_test.values
muestras = []
for _ in range(2000):
    idx = rng.integers(0, len(yt), len(yt))
    ba_final = balanced_accuracy_score(yt[idx], y_pred[idx])
    muestras.append({'Balanced Accuracy': ba_final,
                     'Recall': recall_score(yt[idx], y_pred[idx]),
                     'Especificidad': recall_score(yt[idx], y_pred[idx], pos_label=0),
                     'ROC-AUC': roc_auc_score(yt[idx], puntaje_test[idx]),
                     'Diferencia de Bal. Acc. vs logística': ba_final - balanced_accuracy_score(yt[idx], pred_ref[idx])})
muestras = pd.DataFrame(muestras)

valor_test = {'Balanced Accuracy': metricas_test['Balanced Accuracy'], 'Recall': metricas_test['Recall'],
              'Especificidad': tn_t / (tn_t + fp_t), 'ROC-AUC': metricas_test['ROC-AUC'],
              'Diferencia de Bal. Acc. vs logística': metricas_test['Balanced Accuracy'] - balanced_accuracy_score(y_test, pred_ref)}
intervalos = pd.DataFrame({'Valor en Test': pd.Series(valor_test),
                           'IC 95% inferior': muestras.quantile(0.025),
                           'IC 95% superior': muestras.quantile(0.975)})
print('Modelo:', nombre_mejor)
intervalos.round(4)

### Análisis

Los intervalos de confianza son amplios: la Balanced Accuracy en Test puede estar entre 0.859 y 0.937 y el recall entre 0.797 y 0.940, lo que se debe a que Test solo tiene 80 pacientes con demencia. La especificidad (0.893 a 0.958) y el ROC-AUC (0.936 a 0.979) son más estables. Esto indica que el resultado en Test es consistente con la validación cruzada, pero que diferencias de 1 o 2 puntos entre modelos en Test no son confiables.

Por su parte, la diferencia de Balanced Accuracy frente a la Regresión Logística optimizada es de +0.0185, con un intervalo de 0.004 a 0.037. Como el intervalo no incluye el 0, en Test sí se detecta una ventaja del Stacking; sin embargo, es pequeña y en la validación cruzada repetida fue de solo +0.0033, por lo que el Stacking es ligeramente mejor, pero no por un margen que justifique por sí solo su complejidad.

# Fase 9. Comparación interna

## 16. Comparación de modelos

Una vez seleccionado el modelo final, se evalúan en Test todos los modelos (entrenados con todo Train) para comparar su desempeño y analizar la generalización. Esta evaluación no se usa para elegir ningún modelo.

In [ ]:
modelos_finales = {
    'Dummy': crear_pipeline(DummyClassifier(strategy='most_frequent')),
    'SGD': crear_pipeline(SGDClassifier(random_state=RANDOM_STATE)),
    'Logística sin reg.': crear_pipeline(LogisticRegression(C=1e6, max_iter=5000)),
    'Logística L2': crear_pipeline(LogisticRegression(C=1.0, max_iter=5000)),
    'Logística optimizada': candidatos['Logística optimizada'],
    'SVM lineal': crear_pipeline(LinearSVC(max_iter=10000, random_state=RANDOM_STATE)),
    'SVM polinomial': crear_pipeline(SVC(kernel='poly')),
    'SVM RBF': crear_pipeline(SVC(kernel='rbf')),
    'SVM polinomial optimizado': candidatos['SVM polinomial optimizado'],
    'SVM RBF optimizado': candidatos['SVM RBF optimizado'],
    'Random Forest': crear_pipeline(RandomForestClassifier(random_state=RANDOM_STATE)),
    'Random Forest optimizado': candidatos['Random Forest optimizado'],
    'AdaBoost': crear_pipeline(AdaBoostClassifier(random_state=RANDOM_STATE)),
    'AdaBoost optimizado': candidatos['AdaBoost optimizado'],
    'Gradient Boosting': crear_pipeline(GradientBoostingClassifier(random_state=RANDOM_STATE)),
    'Gradient Boosting optimizado': candidatos['Gradient Boosting optimizado'],
    'XGBoost': crear_pipeline(XGBClassifier(random_state=RANDOM_STATE, n_jobs=1, eval_metric='logloss')),
    'XGBoost optimizado': candidatos['XGBoost optimizado'],
    'Voting duro': candidatos['Voting duro'],
    'Voting suave': candidatos['Voting suave'],
    'Stacking': candidatos['Stacking'],
}

filas = []
metricas_test_modelos = {}
for nombre, modelo in modelos_finales.items():
    modelo = clone(modelo).fit(X_train_fe, y_train)
    pred = modelo.predict(X_test_fe)
    mt = calcular_metricas(y_test, pred, obtener_puntaje(modelo, X_test_fe))
    metricas_test_modelos[nombre] = mt
    mcv = medias_cv(nombre)
    filas.append({'Modelo': nombre,
                  'Bal. Acc. CV': mcv['balanced_accuracy'], 'Bal. Acc. Test': mt['Balanced Accuracy'],
                  'ROC-AUC CV': mcv['roc_auc'], 'ROC-AUC Test': mt['ROC-AUC'],
                  'Recall CV': mcv['recall'], 'Recall Test': mt['Recall'],
                  'F1 CV': mcv['f1'], 'F1 Test': mt['F1'], 'Accuracy Test': mt['Accuracy']})

# Si se adoptó un umbral nuevo, se agrega el modelo final con ese umbral
if umbral_final != umbral:
    filas.append({'Modelo': f'{nombre_mejor} (umbral ajustado)',
                  'Bal. Acc. CV': metricas_oof_final['Balanced Accuracy'], 'Bal. Acc. Test': metricas_test['Balanced Accuracy'],
                  'ROC-AUC CV': metricas_oof_final['ROC-AUC'], 'ROC-AUC Test': metricas_test['ROC-AUC'],
                  'Recall CV': metricas_oof_final['Recall'], 'Recall Test': metricas_test['Recall'],
                  'F1 CV': metricas_oof_final['F1'], 'F1 Test': metricas_test['F1'],
                  'Accuracy Test': metricas_test['Accuracy']})

tabla_final = pd.DataFrame(filas).set_index('Modelo')
tabla_final.round(4)

In [ ]:
orden = tabla_final.sort_values('Bal. Acc. Test').index
y_pos = np.arange(len(orden))
alto = 0.38

fig, ax = plt.subplots(figsize=(9, 10))
ax.barh(y_pos + alto / 2, tabla_final.loc[orden, 'ROC-AUC Test'], alto * 0.9, color=COLORES[0], label='ROC-AUC')
ax.barh(y_pos - alto / 2, tabla_final.loc[orden, 'Bal. Acc. Test'], alto * 0.9, color=COLORES[1], label='Balanced Accuracy')
for i, nombre in enumerate(orden):
    ax.text(tabla_final.loc[nombre, 'ROC-AUC Test'] + 0.01, i + alto / 2,
            f"{tabla_final.loc[nombre, 'ROC-AUC Test']:.3f}", va='center', fontsize=8, color=COLOR_TEXTO)
    ax.text(tabla_final.loc[nombre, 'Bal. Acc. Test'] + 0.01, i - alto / 2,
            f"{tabla_final.loc[nombre, 'Bal. Acc. Test']:.3f}", va='center', fontsize=8, color=COLOR_TEXTO)
ax.set_yticks(y_pos)
ax.set_yticklabels(orden)
ax.set_xlim(0, 1.1)
ax.set_xlabel('Valor en Test')
ax.grid(axis='y', visible=False)
ax.legend(frameon=False, loc='lower right')
ax.set_title('Desempeño de todos los modelos en Test')
plt.tight_layout()
guardar_figura(fig, 'comparacion_modelos_test')
plt.show()

In [ ]:
claves = list(metricas_test)
valores_oof = [metricas_oof_final[k] for k in claves]
valores_test = [metricas_test[k] for k in claves]

x = np.arange(len(claves))
ancho = 0.38
fig, ax = plt.subplots(figsize=(10, 4.5))
b1 = ax.bar(x - ancho / 2, valores_oof, ancho * 0.92, color=COLORES[0], label='OOF (Train)')
b2 = ax.bar(x + ancho / 2, valores_test, ancho * 0.92, color=COLORES[1], label='Test')
for barras in (b1, b2):
    for b in barras:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f'{b.get_height():.3f}',
                ha='center', fontsize=8, color=COLOR_TEXTO)
ax.set_xticks(x)
ax.set_xticklabels(claves)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Valor')
ax.grid(axis='x', visible=False)
ax.legend(frameon=False, ncol=2, loc='upper left')
ax.set_title(f'Generalización: OOF vs Test — {nombre_mejor}')
plt.tight_layout()
plt.show()

### Preguntas

1. ¿La optimización produjo una mejora relevante?<br>
En validación cruzada sí, en Balanced Accuracy y recall: todos los modelos optimizados mejoraron respecto a sus valores por defecto, por ejemplo el SVM polinomial de 0.841 a 0.901, AdaBoost de 0.841 a 0.878, Gradient Boosting de 0.864 a 0.889, XGBoost de 0.863 a 0.888 y la Regresión Logística de 0.868 a 0.893. Sin embargo, el ROC-AUC casi no cambió (entre 0.93 y 0.95 antes y después), lo que indica que la mejora viene principalmente del tratamiento del desbalance (class_weight, el balanceo de Gradient Boosting y scale_pos_weight), que mueve la frontera para detectar más pacientes con demencia, y no de una mejor separación entre clases. En Test la mejora es menos clara: algunos modelos mejoraron (XGBoost de 0.873 a 0.909 y el SVM polinomial de 0.857 a 0.892), mientras que la Regresión Logística optimizada bajó respecto a la L2 por defecto (0.882 contra 0.899), ya que su mayor recall vino acompañado de más falsos positivos.

2. ¿Qué kernel obtuvo mejor desempeño?<br>
Los dos kernels optimizados quedaron prácticamente empatados. El polinomial de grado 2 tuvo la mejor Balanced Accuracy en la validación cruzada de 5 folds (0.901) y en la repetida (0.8916 contra 0.8903), además del mejor ROC-AUC en Test (0.962); el RBF tuvo la mejor Balanced Accuracy en Test (0.901 contra 0.892) y el mayor recall (0.875 contra 0.850). Por tener el mejor resultado en validación cruzada, el SVM polinomial fue el que se usó como modelo base de los ensambles.

3. ¿La complejidad adicional produjo una mejora suficiente?<br>
No. El modelo más complejo, el Stacking, supera a la Regresión Logística optimizada por solo +0.0033 de Balanced Accuracy en la validación cruzada repetida, menos que la desviación entre particiones (≈ 0.02), y los SVM optimizados quedan en una situación similar (+0.0028 y +0.0015). Los modelos de árboles, a pesar de su capacidad para captar relaciones no lineales, quedaron incluso por debajo de la logística (de −0.010 con XGBoost a −0.026 con AdaBoost). Por ende, la complejidad adicional no se tradujo en una mejora sustancial, lo que concuerda con que la relación entre las variables y el diagnóstico es principalmente lineal.

4. ¿Los resultados de Test son consistentes con Cross-Validation?<br>
Sí. En el Stacking, la Balanced Accuracy pasó de 0.897 (OOF) a 0.901 (Test), el recall de 0.878 a 0.875, el F1 de 0.822 a 0.833 y el ROC-AUC de 0.948 a 0.960. Sin embargo, el orden de los modelos en Test no coincide del todo con el de la validación cruzada: XGBoost optimizado obtuvo la mayor Balanced Accuracy en Test (0.909) a pesar de quedar en séptimo lugar en la validación cruzada repetida, y la Regresión Logística optimizada bajó de 0.893 a 0.882. Esto se debe a que Test solo tiene 80 pacientes con demencia y cada uno mueve el recall 1.25 puntos, por lo que las diferencias pequeñas entre modelos en Test no son confiables (sección 15.1).

5. ¿Existe evidencia de underfitting u overfitting?<br>
En el modelo seleccionado no: la brecha entre Train y validación del Stacking es de 0.018 y sus resultados en Test son consistentes con la validación cruzada. Sí hubo overfitting en los modelos de árboles con valores por defecto (brechas de 0.142 en Random Forest, 0.137 en XGBoost y 0.100 en Gradient Boosting), que la optimización redujo (0.070, 0.104 y 0.053), aunque XGBoost conserva la brecha más alta. También apareció overfitting en el SVM RBF con C ≥ 10 o γ ≥ 1 y en el polinomial con grado ≥ 3, y underfitting con C y γ muy pequeños (Balanced Accuracy de 0.500) y en AdaBoost con learning_rate bajo.

6. ¿Qué efecto tuvieron C, γ, degree o coef0?<br>
C controló el equilibrio entre underfitting y overfitting, con el mejor valor en un punto intermedio (C = 1 en ambos kernels). Un γ alto (≥ 1) llevó al RBF a memorizar Train, un degree mayor redujo el desempeño y aumentó la brecha, y coef0 > 0 fue necesario para conservar los términos de menor grado. Sin embargo, el hiperparámetro con más efecto fue class_weight, ya que todas las mejores combinaciones de ambos kernels usan "balanced".

7. ¿Qué modelo elegiría finalmente?<br>
Por el criterio definido antes de ver Test (mayor Balanced Accuracy promedio en la validación cruzada repetida), el Stacking: tiene el mejor resultado en la validación cruzada repetida (0.8922), generaliza bien en Test (Balanced Accuracy 0.901, recall 0.875, especificidad 0.927 y ROC-AUC 0.960) y su brecha es pequeña. No se eligió XGBoost aunque obtuvo el mejor resultado en Test, ya que Test no se usa para seleccionar modelos. Como la ventaja del Stacking sobre la Regresión Logística optimizada es pequeña, la logística sería una alternativa válida si se priorizara la simplicidad y la interpretabilidad.

# Fase 10. Interpretabilidad del modelo final

## 17. Explicación de las predicciones con SHAP

SHAP (*SHapley Additive exPlanations*; Lundberg y Lee, 2017) explica cada predicción individual. Para un paciente, toma el puntaje promedio del modelo (valor base) y reparte entre las variables la diferencia hasta el puntaje de ese paciente: un valor SHAP positivo indica que esa variable empujó la predicción hacia Dementia, y uno negativo, hacia CN. La suma de los valores SHAP de un paciente más el valor base es exactamente su puntaje.

- Los valores se basan en los valores de Shapley de la teoría de juegos cooperativos, que reparten una ganancia de forma justa entre los jugadores; aquí los jugadores son las variables.
- Se usa el explicador por permutación (`PermutationExplainer`), que funciona con cualquier modelo, porque el modelo final puede ser un SVM, un ensamble de árboles o un Voting/Stacking.
- Se explican los 325 pacientes de Test con 100 pacientes de Train como referencia (fondo). Esto no influye en ninguna decisión: el modelo ya está elegido y entrenado.
- SHAP explica lo que hace el modelo, no la causa biológica de la enfermedad.

A diferencia de la importancia por permutación de la sección 12.5, que es global (qué variables importan en general), SHAP indica también la **dirección** del efecto y permite explicar a un paciente concreto.

En la gráfica de enjambre (*beeswarm*), cada punto es un paciente de Test: su posición horizontal es el valor SHAP de esa variable y el color indica si el valor de la variable es alto (rojo) o bajo (azul). En la gráfica de cascada (*waterfall*) se parte del valor base y cada barra suma o resta la contribución de una variable hasta llegar al puntaje del paciente.

In [ ]:
columnas_fe = X_train_fe.columns.tolist()
nombres_shap = [etiquetas_fe[c] for c in columnas_fe]


def a_numerico(datos):
    # SHAP trabaja con números: Sexo pasa a 1 = Masculino, 0 = Femenino
    datos = datos.copy()
    datos['Sexo'] = (datos['Sexo'] == 'Masculino').astype(float)
    return datos.astype(float)


def puntaje_desde_numerico(arreglo):
    datos = pd.DataFrame(arreglo, columns=columnas_fe)
    datos['Sexo'] = np.where(datos['Sexo'] >= 0.5, 'Masculino', 'Femenino')
    return obtener_puntaje(best_model, datos)


fondo = a_numerico(X_train_fe.sample(100, random_state=RANDOM_STATE))
explicador = shap.PermutationExplainer(puntaje_desde_numerico,
                                       shap.maskers.Independent(fondo.values, max_samples=100),
                                       feature_names=nombres_shap)

inicio = time.perf_counter()
valores_shap = explicador(a_numerico(X_test_fe).values, max_evals=10 * (2 * len(columnas_fe) + 1), silent=True)
print(f'Valores SHAP calculados para {len(X_test_fe)} pacientes de Test en {time.perf_counter() - inicio:.0f} s')
print('Puntaje explicado:', etiquetas_puntaje[tipo_puntaje(best_model)])
print('Valor base (puntaje promedio del fondo):', round(float(np.mean(valores_shap.base_values)), 4))

In [ ]:
shap.plots.beeswarm(valores_shap, max_display=len(columnas_fe), show=False,
                    color_bar_label='Valor de la variable')
fig = plt.gcf()
fig.set_size_inches(9, 5.5)
fig.axes[0].set_xlabel('Valor SHAP (contribución a la función de decisión)')
fig.axes[-1].set_yticklabels(['Bajo', 'Alto'])   # barra de color
plt.title(f'Valores SHAP en Test — {nombre_mejor}')
plt.tight_layout()
guardar_figura(fig, 'shap_beeswarm')
plt.show()

In [ ]:
importancia_shap = pd.Series(np.abs(valores_shap.values).mean(axis=0), index=nombres_shap)
comparacion_importancia = pd.DataFrame({
    'Media |SHAP| (modelo final, Test)': importancia_shap,
    'Lugar SHAP': importancia_shap.rank(ascending=False).astype(int),
    'Permutación (Random Forest, CV)': importancias['Permutación (media)'],
    'Lugar permutación': importancias['Permutación (media)'].rank(ascending=False).astype(int),
}).sort_values('Lugar SHAP')
comparacion_importancia.round(4)

### Explicación de pacientes individuales

Se explican dos pacientes de Test con demencia: el que el modelo detectó con mayor seguridad (verdadero positivo) y, si existe, el falso negativo con el puntaje más bajo, es decir, el caso que el modelo más claramente confundió con una persona sana.

In [ ]:
posiciones_dem = np.where(y_test.values == 1)[0]
paciente_tp = posiciones_dem[np.argmax(puntaje_test[posiciones_dem])]
falsos_neg = np.where((y_test.values == 1) & (y_pred == 0))[0]
pacientes = [('Verdadero positivo', paciente_tp)]
if len(falsos_neg) > 0:
    pacientes.append(('Falso negativo', falsos_neg[np.argmin(puntaje_test[falsos_neg])]))

for etiqueta, pos in pacientes:
    print(f'{etiqueta}: paciente en la posición {pos} de Test | puntaje {puntaje_test[pos]:.3f} (umbral {umbral_final:.3f})')
    shap.plots.waterfall(valores_shap[pos], max_display=len(columnas_fe), show=False)
    fig = plt.gcf()
    plt.title(f'{etiqueta} — {nombre_mejor}')
    plt.tight_layout()
    guardar_figura(fig, 'shap_' + etiqueta.lower().replace(' ', '_'))
    plt.show()

### Análisis

Según SHAP, las variables con más peso en el Stacking son el ventrículo lateral inferior / ICV (media de |SHAP| de 0.81), el hipocampo / ICV (0.66), APOE4 (0.54) y la corteza entorrinal / ICV (0.43), mientras que Sexo es la que menos aporta (0.07). La dirección es coherente con la enfermedad: un ventrículo lateral inferior grande, un hipocampo, una corteza entorrinal y una amígdala pequeños, un grosor cortical menor y más copias de APOE4 empujan la predicción hacia Dementia, y una escolaridad alta empuja hacia CN. En cambio, la edad alta y los ventrículos laterales grandes empujan hacia CN, lo cual no significa que protejan contra la demencia: el modelo evalúa cada variable junto con las demás, por lo que, con el mismo volumen cerebral, en una persona mayor la atrofia es más esperada para su edad, y los ventrículos laterales están correlacionados con el ventrículo lateral inferior, que ya aporta esa información.

Esto coincide con la importancia por permutación de Random Forest en las dos variables más importantes (ventrículo lateral inferior e hipocampo) y en la menos importante (Sexo), y con el EDA, donde estas variables tienen la mayor correlación con el diagnóstico. La diferencia principal es APOE4, que pasa del quinto lugar en Random Forest al tercero en el Stacking, y Escolaridad (del noveno al séptimo), posiblemente debido a que el metamodelo da más peso a la Regresión Logística, que usa estas variables de forma lineal.

El falso negativo con el puntaje más bajo (−2.37) es una paciente de 76.7 años cuyo hipocampo / ICV (5.15), corteza entorrinal / ICV (2.99) y amígdala / ICV (2.13) empujan la predicción hacia CN, es decir, el modelo los interpreta como volúmenes de una persona sana; además, no tiene dato de APOE4 (se imputó con MICE) y tiene 16 años de escolaridad. Solo el ventrículo lateral inferior empuja ligeramente hacia Dementia (+0.15). Por ende, este error no se corregiría moviendo el umbral, ya que con las 11 características del modelo la paciente se ve como una persona cognitivamente normal; una explicación posible es que su diagnóstico dependa de información que el modelo no tiene, como las pruebas cognitivas o los biomarcadores de amiloide.

# VI. Comparación con la literatura

# Fase 11. Comparación con la literatura

## 18. Comparación externa

Se seleccionaron artículos científicos que usan la misma base de datos (ADNI) y la misma tarea (CN vs demencia por Alzheimer): tres con SVM (Cuingnet et al., 2011; Zhang et al., 2011; Samper-González et al., 2018), dos recientes con volúmenes cerebrales (Zarei et al., 2024; Altun y Cetinok, 2026) y uno con ensambles de árboles, SVM y SHAP sobre volúmenes de FreeSurfer y APOE4 (Bloch y Friedrich, 2022). Como los artículos reportan sensibilidad (recall), especificidad, accuracy, balanced accuracy y AUC, para este trabajo se calculan las mismas métricas en Test.

In [ ]:
especificidad = tn_t / (tn_t + fp_t)
resultado_propio = (f"Sens. {metricas_test['Recall']:.1%}, Espec. {especificidad:.1%}, "
                    f"Acc. {metricas_test['Accuracy']:.1%}, Bal. Acc. {metricas_test['Balanced Accuracy']:.3f}, "
                    f"AUC {metricas_test['ROC-AUC']:.3f}")

literatura = pd.DataFrame([
    {'Trabajo': 'Este trabajo',
     'Dataset': f'ADNI (fases 1 a 4): {(y == 0).sum()} CN + {(y == 1).sum()} Dementia',
     'Modelo': f'{nombre_mejor} ({X_train_fe.shape[1]} características: volúmenes FreeSurfer/ICV, APOE4, edad, sexo, escolaridad)',
     'Protocolo experimental': 'Train/Test 80/20 estratificado; CV estratificada de 5 folds en Train para selección y optimización',
     'Métrica(s)': 'Sensibilidad, especificidad, accuracy, balanced accuracy, ROC-AUC (Test)',
     'Resultado': resultado_propio},
    {'Trabajo': 'Cuingnet et al. (2011)',
     'Dataset': 'ADNI: 162 CN + 137 AD',
     'Modelo': '10 métodos de extracción de características de MRI (vóxeles y grosor cortical con SVM lineal; volumen de hipocampo con ventana de Parzen)',
     'Protocolo experimental': 'Train/Test 50/50 (81 CN + 69 AD / 81 CN + 68 AD), balanceado por edad y sexo',
     'Métrica(s)': 'Sensibilidad y especificidad (Test)',
     'Resultado': 'Mejor método (vóxeles): Sens. 81%, Espec. 95%. Volumen de hipocampo con FreeSurfer: Sens. 63%, Espec. 80%'},
    {'Trabajo': 'Zhang et al. (2011)',
     'Dataset': 'ADNI: 52 CN + 51 AD',
     'Modelo': 'SVM multi-kernel con kernel lineal (C = 1); MRI (93 volúmenes regionales), FDG-PET y LCR',
     'Protocolo experimental': 'CV de 10 folds repetida 10 veces, sin Test independiente',
     'Métrica(s)': 'Accuracy, sensibilidad, especificidad, AUC',
     'Resultado': 'Solo MRI: Acc. 86.2%, Sens. 86%, Espec. 86.3%. MRI + PET + LCR: Acc. 93.2%, Sens. 93%, Espec. 93.3%, AUC 0.976'},
    {'Trabajo': 'Samper-González et al. (2018)',
     'Dataset': 'ADNI: 282 CN + 237 AD',
     'Modelo': 'SVM lineal (también Regresión Logística L2 y Random Forest); MRI T1 por vóxeles o por regiones (atlas AAL2)',
     'Protocolo experimental': '250 particiones aleatorias estratificadas; CV interna de 10 folds para hiperparámetros',
     'Métrica(s)': 'Balanced accuracy',
     'Resultado': 'Vóxeles: 0.87 ± 0.026. Regiones: 0.84 ± 0.024. SVM lineal ≈ Regresión Logística L2 > Random Forest'},
    {'Trabajo': 'Zarei et al. (2024)',
     'Dataset': 'ADNI: 200 CN + 199 AD (853 y 602 imágenes)',
     'Modelo': 'SVM con características radiómicas de hipocampo y amígdala (1,746 por región); también CNN 3D y su fusión',
     'Protocolo experimental': '5 repeticiones de partición aleatoria 80/20, evaluación por imagen',
     'Métrica(s)': 'Accuracy y AUC',
     'Resultado': 'SVM radiómico: Acc. 82.6% ± 3.4, AUC 0.888 ± 0.036. Radiómica + CNN: Acc. 84.4%, AUC 0.900'},
    {'Trabajo': 'Altun y Cetinok (2026)',
     'Dataset': 'ADNI: 365 CN + 153 AD',
     'Modelo': 'Regresión Logística ElasticNet; volúmenes y medidas de FastSurfer + edad, sexo, escolaridad y APOE4',
     'Protocolo experimental': 'Train/Test 80/20 por paciente; CV de 5 folds en Train para hiperparámetros',
     'Métrica(s)': 'Accuracy, balanced accuracy, macro-F1, ROC-AUC (Test)',
     'Resultado': 'Solo morfometría: Acc. 0.856, Bal. Acc. 0.823, AUC 0.901. Con demografía y APOE4 (sin MMSE): Acc. 0.865, Bal. Acc. 0.858, AUC 0.900'},
    {'Trabajo': 'Bloch y Friedrich (2022)',
     'Dataset': 'ADNI: 512 CN + 335 AD (de 1,700 sujetos CN, MCI y AD en la visita inicial)',
     'Modelo': 'Random Forest, XGBoost, SVM (polinomial y RBF), Regresión Logística y árbol de decisión; 152 medidas de FreeSurfer normalizadas por eTIV + edad, sexo, escolaridad y APOE4; explicación con SHAP',
     'Protocolo experimental': 'Train/Test 80/20 estratificado por paciente; optimización bayesiana con CV 10×10 en Train; validación externa en AIBL y OASIS',
     'Métrica(s)': 'Accuracy, balanced accuracy, AUROC, F1 y MCC (Test de ADNI)',
     'Resultado': 'Con APOE4 y sin selección de variables: RF Bal. Acc. 89.1%, AUC 0.965; XGBoost 89.3%, 0.951; SVM RBF 86.9%, 0.962; Logística 87.1%, 0.965. Al agregar pruebas cognitivas (MMSE y memoria lógica) llegan a 97–100%'},
]).set_index('Trabajo')

literatura

### Discusión

1. ¿Sus resultados son comparables con los reportados en la literatura?<br>
Parcialmente. En Test se obtuvo una sensibilidad de 87.5%, una especificidad de 92.7%, una balanced accuracy de 0.901 y un AUC de 0.960. Son resultados similares o superiores a los de los trabajos que usan solo resonancia magnética: el mejor método de Cuingnet et al. tiene una balanced accuracy de ≈ 0.88 (81% y 95%) y su método con volumen de hipocampo ≈ 0.72, Zhang et al. reportan ≈ 0.86 con solo MRI, Samper-González et al. de 0.84 a 0.87, Zarei et al. una accuracy de 82.6% a 84.4% y Altun y Cetinok una balanced accuracy de 0.858 con demografía y APOE4. El trabajo más comparable es el de Bloch y Friedrich, que usa volúmenes de FreeSurfer, APOE4 y ensambles de árboles, y obtiene balanced accuracies de 86.9% a 89.3% y AUC de 0.951 a 0.965, prácticamente iguales a las de este trabajo y dentro de su intervalo de confianza (0.859 a 0.937). Los resultados quedan por debajo del modelo multimodal de Zhang et al. (accuracy 93.2%, AUC 0.976), que además de MRI usa PET y LCR, y de los modelos de Bloch y Friedrich que agregan pruebas cognitivas (97 a 100%). Sin embargo, como se discute abajo, las condiciones experimentales no son iguales, por lo que no se puede afirmar que este trabajo supere a ninguno de ellos.

2. ¿Los trabajos utilizan el mismo dataset?<br>
Todos usan ADNI, pero no el mismo subconjunto. Los artículos se publicaron entre 2011 y 2026 y usan las fases de ADNI disponibles en ese momento, con sus propios criterios de selección y muestras de 103 a 847 participantes CN y AD. Este trabajo usa 1,624 pacientes de las fases ADNI1, GO, 2, 3 y 4.

3. ¿Utilizan el mismo número de características?<br>
No. Cuingnet et al. usan desde miles de vóxeles hasta solo el volumen del hipocampo; Zhang et al. usan 93 volúmenes regionales (más 93 de PET y 3 de LCR en el modelo multimodal); Samper-González et al. usan vóxeles o regiones de un atlas; Zarei et al. usan 1,746 características radiómicas por región; Altun y Cetinok, medidas morfométricas de FastSurfer junto con edad, sexo, escolaridad y APOE4; y Bloch y Friedrich, 152 medidas de FreeSurfer con las mismas variables demográficas y APOE4. Este trabajo usa solo 11 características, que resumen las regiones más afectadas por la enfermedad.

4. ¿Utilizan la misma partición de datos?<br>
No. Cuingnet et al. dividen 50/50 balanceando edad y sexo; Zhang et al. usan CV de 10 folds repetida; Samper-González et al. usan 250 particiones aleatorias; Zarei et al. usan 5 particiones aleatorias 80/20 por imagen, sin indicar que sean por paciente, aunque tienen más de una imagen por participante (posible fuga de información). Altun y Cetinok y Bloch y Friedrich usan una partición 80/20 por paciente, igual que este trabajo, que usa una partición 80/20 estratificada.

5. ¿Utilizan Cross-Validation o un Test independiente?<br>
Cuingnet et al., Altun y Cetinok y Bloch y Friedrich reportan un Test independiente, y estos últimos agregan una validación externa con las cohortes AIBL y OASIS. Zhang et al. reportan solo validación cruzada, y Samper-González et al. y Zarei et al. reportan el promedio de varias particiones. Este trabajo usa validación cruzada para optimizar y seleccionar (incluida la validación cruzada repetida 5 × 5) y un Test independiente para la evaluación final.

6. ¿Existen diferencias en preprocesamiento?<br>
Sí. Cuingnet et al., Zhang et al. y Samper-González et al. extraen sus características directamente de las imágenes con sus propios procesos (segmentación por vóxeles, atlas de regiones); Zarei et al. usan radiómica y una red convolucional 3D; Altun y Cetinok usan FastSurfer; y Bloch y Friedrich usan FreeSurfer con las medidas normalizadas por el volumen intracraneal (eTIV), de forma similar a este trabajo. Este trabajo usa volúmenes ya calculados con FreeSurfer y armonizados con ComBat (tablas ADSP-PHC), normalizados por ICV, y completa APOE4 con la prueba en plasma (C2N) y con MICE.

7. ¿Es válido comparar directamente las métricas?<br>
No de forma directa. Además de las diferencias en la muestra, las características y el protocolo, las clases están balanceadas de forma distinta: en los artículos la proporción CN/AD va de 50/50 a aproximadamente 70/30, mientras que aquí es 75/25. Por eso la accuracy no es comparable, y la sensibilidad, la especificidad y la balanced accuracy son las métricas más adecuadas para comparar. Además, con 80 pacientes con demencia en Test, la balanced accuracy de este trabajo tiene un intervalo de confianza de ± 4 puntos aproximadamente.

8. ¿Qué diferencias metodológicas podrían explicar resultados superiores o inferiores?<br>
Pueden explicar resultados superiores: el uso de variables que no son de imagen (APOE4, edad y escolaridad), una muestra más grande (1,624 pacientes contra 103 a 847), volúmenes armonizados con ComBat y normalizados por ICV que reducen las diferencias entre escáneres y tamaños de cabeza, y el tratamiento del desbalance. Pueden explicar resultados inferiores: no usar biomarcadores de PET o LCR ni pruebas cognitivas (que explican la ventaja del modelo multimodal de Zhang et al. y de los modelos de Bloch y Friedrich con pruebas cognitivas) y usar 11 características resumidas en lugar de la imagen completa. Además, este trabajo evalúa en una sola partición y sin validación externa, mientras que Zhang et al., Samper-González et al. y Zarei et al. promedian varias particiones y Bloch y Friedrich validan en otras cohortes, por lo que su estimación es menos variable.

# VII. Conclusiones

Se cumplió el objetivo del proyecto: se compararon todos los modelos de clasificación vistos en la materia (Regresión Logística, SGD, SVM con kernels lineal, polinomial y RBF, Random Forest, AdaBoost, Gradient Boosting, XGBoost, Voting y Stacking) con los mismos datos, el mismo Pipeline y las mismas particiones, para distinguir a personas cognitivamente normales de personas con demencia. Por el criterio definido antes de ver Test (mayor Balanced Accuracy promedio en la validación cruzada repetida 5 × 5), se seleccionó el Stacking, que obtuvo 0.892 en la validación cruzada repetida y, en Test, una Balanced Accuracy de 0.901 (IC 95%: 0.859 a 0.937), un recall de 0.875 (70 de 80 pacientes con demencia), una especificidad de 0.927 y un ROC-AUC de 0.960.

Sin embargo, el hallazgo principal es que la complejidad adicional casi no mejoró el resultado. El Stacking supera a la Regresión Logística optimizada por solo +0.003 en la validación cruzada repetida, y ningún modelo de árboles, ni por bagging (Random Forest) ni por boosting (AdaBoost, Gradient Boosting y XGBoost), la superó. Esto, junto con que el SVM polinomial eligió el grado más bajo y que los kernels no lineales quedaron prácticamente igual que el modelo lineal, indica que la relación entre los biomarcadores y el diagnóstico es principalmente lineal. El único modelo que superó al mejor modelo individual, aunque por muy poco, fue el Stacking, posiblemente debido a que combina modelos de familias distintas que se equivocan de forma diferente.

El tratamiento del desbalance (75% CN y 25% Dementia) fue la decisión con mayor efecto en todo el proyecto. Al optimizar con Balanced Accuracy, class_weight = "balanced", el balanceo por paciente de Gradient Boosting y scale_pos_weight en XGBoost aumentaron el recall de los modelos sin cambiar prácticamente el ROC-AUC, es decir, movieron la frontera para detectar más pacientes con demencia. Por la misma razón, el ajuste del umbral ya no aportó una mejora relevante (+0.0015) y se conservó el umbral por defecto.

Según SHAP, las predicciones se basan principalmente en el ventrículo lateral inferior, el hipocampo, APOE4 y la corteza entorrinal, lo cual concuerda con el EDA, con la importancia por permutación de Random Forest y con la fisiopatología de la enfermedad, ya que el lóbulo temporal medial es una de las primeras regiones que se atrofian. En comparación con la literatura, los resultados son similares a los de Bloch y Friedrich (2022), el trabajo más comparable; sin embargo, las diferencias en los subconjuntos de ADNI, las características y los protocolos de validación no permiten afirmar que este trabajo supere a los trabajos previos.

Las principales limitaciones son que solo se clasifica CN contra Dementia (sin MCI, la etapa en la que un diagnóstico temprano sería más útil), que se usa una sola cohorte y una sola partición de Test con 80 pacientes con demencia (de ahí la amplitud de los intervalos de confianza), que los datos corresponden a un solo momento por paciente y que los volúmenes están armonizados con ComBat, por lo que un paciente nuevo también tendría que procesarse de la misma forma. Como trabajo futuro se propone incluir a los pacientes con MCI, validar el modelo en cohortes externas como AIBL u OASIS y agregar biomarcadores de plasma, LCR o PET.

# VIII. Referencias

- Alzheimer's Disease Neuroimaging Initiative (ADNI). LONI Image & Data Archive. https://adni.loni.usc.edu
- Cuingnet, R., Gerardin, E., Tessieras, J., Auzias, G., Lehéricy, S., Habert, M.-O., Chupin, M., Benali, H., Colliot, O., & the Alzheimer's Disease Neuroimaging Initiative. (2011). Automatic classification of patients with Alzheimer's disease from structural MRI: A comparison of ten methods using the ADNI database. *NeuroImage, 56*(2), 766–781. https://doi.org/10.1016/j.neuroimage.2010.06.013
- Zhang, D., Wang, Y., Zhou, L., Yuan, H., Shen, D., & the Alzheimer's Disease Neuroimaging Initiative. (2011). Multimodal classification of Alzheimer's disease and mild cognitive impairment. *NeuroImage, 55*(3), 856–867. https://doi.org/10.1016/j.neuroimage.2011.01.008
- Samper-González, J., Burgos, N., Bottani, S., Fontanella, S., Lu, P., Marcoux, A., Routier, A., Guillon, J., Bacci, M., Wen, J., Bertrand, A., Bertin, H., Habert, M.-O., Durrleman, S., Evgeniou, T., & Colliot, O. (2018). Reproducible evaluation of classification methods in Alzheimer's disease: Framework and application to MRI and PET data. *NeuroImage, 183*, 504–521. https://doi.org/10.1016/j.neuroimage.2018.08.042
- Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.
- Scikit-learn. Support Vector Machines. https://scikit-learn.org/stable/modules/svm.html
- Scikit-learn. IterativeImputer. https://scikit-learn.org/stable/modules/generated/sklearn.impute.IterativeImputer.html
- Zarei, A., Keshavarz, A., Jafari, E., Nemati, R., Farhadi, A., Gholamrezanezhad, A., Rostami, H., & Assadi, M. (2024). Automated classification of Alzheimer's disease, mild cognitive impairment, and cognitively normal patients using 3D convolutional neural network and radiomic features from T1-weighted brain MRI: A comparative study on detection accuracy. *Clinical Imaging, 115*, 110301. https://doi.org/10.1016/j.clinimag.2024.110301
- Altun, S. N., & Cetinok, H. (2026). FastSurfer-Based Brain Morphometry and Machine-Learning Classification Across the Alzheimer's Disease Spectrum. *Brain Sciences, 16*(9), 980. https://doi.org/10.3390/brainsci16090980
- Bloch, L., Friedrich, C. M., & the Alzheimer's Disease Neuroimaging Initiative. (2022). Machine learning workflow to explain black-box models for early Alzheimer's disease classification evaluated for multiple datasets. *SN Computer Science, 3*(6), 509. https://doi.org/10.1007/s42979-022-01371-y
- Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32. https://doi.org/10.1023/A:1010933404324
- Freund, Y., & Schapire, R. E. (1997). A decision-theoretic generalization of on-line learning and an application to boosting. *Journal of Computer and System Sciences, 55*(1), 119–139. https://doi.org/10.1006/jcss.1997.1504
- Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine. *The Annals of Statistics, 29*(5), 1189–1232. https://doi.org/10.1214/aos/1013203451
- Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785–794. https://doi.org/10.1145/2939672.2939785
- Wolpert, D. H. (1992). Stacked generalization. *Neural Networks, 5*(2), 241–259. https://doi.org/10.1016/S0893-6080(05)80023-1
- Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. *Advances in Neural Information Processing Systems, 30*.
- Scikit-learn. Ensembles: Gradient boosting, random forests, bagging, voting, stacking. https://scikit-learn.org/stable/modules/ensemble.html
- XGBoost. XGBoost Documentation. https://xgboost.readthedocs.io

# IX. Anexo: exportación de los modelos para la interfaz

Se guardan con `joblib` los modelos que usará la interfaz en Streamlit: el mejor de cada familia, los dos ensambles mixtos con puntaje (Voting suave y Stacking) y el modelo final. Cada archivo contiene el **Pipeline completo** (imputación con MICE, escalamiento, codificación de Sexo y modelo), por lo que la interfaz solo tiene que aplicar `crear_features` a los datos del paciente y llamar al modelo.

- Todos los modelos se entrenan **solo con Train**, igual que en la evaluación, para que la interfaz muestre exactamente los modelos cuyos resultados se reportan.
- El SVM se guarda con `probability=True` para que la interfaz pueda mostrar una probabilidad; la clase se sigue decidiendo con la distancia a la frontera.
- `metadata.json` guarda las variables, sus rangos en Train (para validar lo que se captura), el umbral del modelo final, las métricas de Test y las versiones de las librerías. La interfaz debe usar la **misma versión de scikit-learn** con la que se guardaron los modelos.
- `pacientes_ejemplo.csv` (algunos pacientes de Test) y `fondo_shap.csv` (100 pacientes de Train para SHAP) contienen datos de ADNI: el acuerdo de uso de ADNI no permite redistribuirlos, así que no deben publicarse ni subirse a repositorios públicos.

In [ ]:
CARPETA_MODELOS = RUTA + 'modelos_adni/'
os.makedirs(CARPETA_MODELOS, exist_ok=True)

nombre_boosting = mejor_de({'AdaBoost optimizado': grid_ada, 'Gradient Boosting optimizado': grid_gb,
                            'XGBoost optimizado': grid_xgb})
a_exportar = {
    'logistica': ('Logística optimizada', candidatos['Logística optimizada']),
    'svm': (nombre_svm, clone(candidatos[nombre_svm]).set_params(modelo__probability=True,
                                                                 modelo__random_state=RANDOM_STATE)),
    'random_forest': ('Random Forest optimizado', candidatos['Random Forest optimizado']),
    'boosting': (nombre_boosting, candidatos[nombre_boosting]),
    'voting_suave': ('Voting suave', candidatos['Voting suave']),
    'stacking': ('Stacking', candidatos['Stacking']),
}

info_modelos = {}
for clave, (nombre, modelo) in a_exportar.items():
    modelo = clone(modelo).fit(X_train_fe, y_train)
    joblib.dump(modelo, CARPETA_MODELOS + clave + '.joblib')
    info_modelos[clave] = {'archivo': clave + '.joblib', 'nombre': nombre,
                           'tipo_puntaje': tipo_puntaje(modelo), 'umbral': umbral_por_defecto(modelo),
                           'metricas_test': {k: round(float(v), 4) for k, v in metricas_test_modelos[nombre].items()}}

joblib.dump(best_model, CARPETA_MODELOS + 'modelo_final.joblib')
info_modelos['modelo_final'] = {'archivo': 'modelo_final.joblib', 'nombre': nombre_mejor,
                                'tipo_puntaje': tipo_puntaje(best_model), 'umbral': float(umbral_final),
                                'metricas_test': {k: round(float(v), 4) for k, v in metricas_test.items()}}

numericas = X_train.select_dtypes(include='number').columns
metadata = {
    'variables_originales': X_train.columns.tolist(),
    'variables_modelo': columnas_fe,
    'etiquetas_variables': etiquetas_fe,
    'clases': {'0': 'CN', '1': 'Dementia'},
    'categorias_sexo': sorted(X_train['Sexo'].dropna().unique().tolist()),
    'rangos_train': {c: {'min': float(X_train[c].min()), 'max': float(X_train[c].max()),
                         'mediana': float(X_train[c].median())} for c in numericas},
    'modelos': info_modelos,
    'versiones': {'python': sys.version.split()[0], 'scikit-learn': sklearn.__version__,
                  'xgboost': xgboost.__version__, 'shap': shap.__version__,
                  'numpy': np.__version__, 'pandas': pd.__version__, 'joblib': joblib.__version__},
}
with open(CARPETA_MODELOS + 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# Datos de ADNI de uso local (no publicar): pacientes de ejemplo y fondo para SHAP
ejemplos = X_test.copy()
ejemplos['DX'] = y_test.map({0: 'CN', 1: 'Dementia'})
ejemplos = pd.concat([ejemplos[ejemplos['DX'] == 'CN'].sample(4, random_state=RANDOM_STATE),
                      ejemplos[ejemplos['DX'] == 'Dementia'].sample(4, random_state=RANDOM_STATE)])
ejemplos.to_csv(CARPETA_MODELOS + 'pacientes_ejemplo.csv', index=False)
X_train.loc[fondo.index].to_csv(CARPETA_MODELOS + 'fondo_shap.csv', index=False)

print('Archivos guardados en', CARPETA_MODELOS)
for archivo in sorted(os.listdir(CARPETA_MODELOS)):
    print(f'  {archivo:<26} {os.path.getsize(CARPETA_MODELOS + archivo) / 1024:,.0f} KB')
print('\nVersiones:', metadata['versiones'])